# 04 - Clinically relevant and pharmacogenomic variation

![cohort](https://img.shields.io/badge/cohort-KZ--219-1b9e77)
![platform](https://img.shields.io/badge/platform-DNBSEQ--G400-7570b3)
![pipeline](https://img.shields.io/badge/DRAGEN-v3.10.16-d95f02)
![reference](https://img.shields.io/badge/reference-GRCh37%2Fhg19-e7298a)
![annotation](https://img.shields.io/badge/ANNOVAR-v2025Mar02-666666)
![bioproject](https://img.shields.io/badge/BioProject-PRJNA1224597-333333)

| | |
|---|---|
| **Cohort** | KZ-219 - 219 healthy Kazakh adults |
| **Sequencing** | DNBSEQ-G400, PCR-free, 100 bp paired-end |
| **Alignment and variant calling** | Illumina DRAGEN Bio-IT Platform v3.10.16 |
| **Reference** | GRCh37 / hg19 (DRAGEN hash table v8) |
| **Annotation** | ANNOVAR v2025Mar02 |
| **Raw data** | NCBI SRA, BioProject PRJNA1224597 |

**Pre-specified exclusions**

| Excluded | Reason |
|---|---|
| Variants inside known pathogenic repeat tracts | short reads cannot size a repeat |
| Variants in genes with a high-identity paralogue or pseudogene | read assignment is unreliable |
| Star alleles and diplotypes | haplotypes are not phased in this dataset |
| HLA alleles | tag SNPs are not HLA alleles; direct typing is out of scope |
| ClinVar records below `MIN_CLINVAR_STARS` review status | single-submitter assertions are not used for population reporting |


## 1. Configuration and registries

Curated constants: preferred clinical transcripts, curated HGVS with the GRCh37 alleles each
entry describes, repeat tracts, paralogue-bearing genes, effect-allele polarisation and the
ACMG SF v3.3 gene list with its reporting footnotes.

In [ ]:
import gc
import json
import os
import platform
import re
import shutil
import subprocess
import sys
from datetime import datetime

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.ticker import MaxNLocator
import numpy as np
import pandas as pd
import scipy
from IPython.display import display
from tqdm import tqdm

INPUT_FILE = os.environ.get('KZ_KNOWN_FILE',
                            'work/03_af/kz_219_hg19_AF_from_GT_known.txt') 
SLIM_FILE = INPUT_FILE.replace('.txt', '.slim.txt')
CKPT_FILE = INPUT_FILE.replace('.txt', '.filtered.parquet')
OUT_DIR = os.environ.get('KZ_OUT_DIR', 'results/04_biomed')

COHORT = 'KZ-219'
N_COHORT = 219
N_FEMALE, N_MALE = 145, 74
N_CHR_KZ = 2 * N_COHORT
AF_COL = 'AF_pass'

CLINVAR_RELEASE = 'clinvar_20250721'
ANNOVAR_BUILD = 'ANNOVAR 2025Mar02, refGene'
CALLER_SAMPLE = 'DRAGEN v3.10.16'
CALLER_JOINT = 'DRAGEN v4.3.6'

MIN_CLINVAR_STARS = 2
MIN_CALL_RATE = 0.90

os.makedirs(OUT_DIR, exist_ok=True)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.float_format', '{:.4f}'.format)

# ClinVar
PLP_TOKENS = {'pathogenic', 'likely_pathogenic'}
EXCL_TOKENS = {'benign', 'likely_benign', 'uncertain_significance', 'conflicting'}
SIDE_TOKENS = {'risk_factor', 'drug_response', 'association', 'protective', 'affects'}

STAR_MAP = {
    'practice_guideline': 4,
    'reviewed_by_expert_panel': 3,
    'criteria_provided,_multiple_submitters,_no_conflicts': 2,
    'criteria_provided,_multiple_submitters': 2,
    'criteria_provided,_single_submitter': 1,
    'criteria_provided,_conflicting_interpretations': 1,
    'criteria_provided,_conflicting_classifications': 1,
    'no_assertion_criteria_provided': 0,
    'no_assertion_provided': 0,
    'no_classification_provided': 0,
    'no_classifications_from_unflagged_records': 0,
    'no_classification_for_the_single_variant': 0,
}


def clnsig_tokens(value):
    if not isinstance(value, str) or not value.strip():
        return set()
    return {t.strip().strip('_').lower().replace(' ', '_')
            for t in re.split(r'[|/,]', value) if t.strip()}


def is_plp(value):
    t = clnsig_tokens(value)
    return bool(t) and not (t & EXCL_TOKENS) and bool(t & PLP_TOKENS)


def has_side_assertion(value):
    return bool(clnsig_tokens(value) & SIDE_TOKENS)


def clinvar_stars(revstat):
    if not isinstance(revstat, str):
        return np.nan
    return STAR_MAP.get(revstat.strip().lower(), np.nan)

# excluded regions and genes, hg19 coordinates, padded
REPEAT_TRACTS_HG19 = [
    ('HTT', 'chr4', 3076400, 3076860),
    ('ATXN1', 'chr6', 16327660, 16328100),
    ('ATXN2', 'chr12', 112036550, 112037030),
    ('ATXN3', 'chr14', 92537150, 92537620),
    ('ATXN7', 'chr3', 63898160, 63898600),
    ('ATXN8OS', 'chr13', 70713300, 70713700),
    ('CACNA1A', 'chr19', 13318460, 13318900),
    ('TBP', 'chr6', 170870790, 170871250),
    ('ATN1', 'chr12', 7045680, 7046130),
    ('PPP2R2B', 'chr5', 146258080, 146258530),
    ('C9orf72', 'chr9', 27573280, 27573750),
    ('FMR1', 'chrX', 146993350, 146993830),
    ('AR', 'chrX', 66764950, 66765430),
    ('DMPK', 'chr19', 46273240, 46273700),
    ('FXN', 'chr9', 71651980, 71652450),
    ('CSTB', 'chr21', 45196150, 45196500),
    ('JPH3', 'chr16', 87637700, 87638100),
    ('NOTCH2NLC', 'chr1', 120458000, 120458500),
]

PARALOGUE_GENES = {
    'PMS2': 'PMS2CL pseudogene',
    'GBA': 'GBAP1 pseudogene',
    'GBA1': 'GBAP1 pseudogene',
    'CYP11B1': 'CYP11B2 paralogue',
    'CYP11B2': 'CYP11B1 paralogue',
    'CYP21A2': 'CYP21A1P pseudogene',
    'CYP2D6': 'CYP2D7 and CYP2D8 pseudogenes',
    'SMN1': 'SMN2 paralogue',
    'SMN2': 'SMN1 paralogue',
    'STRC': 'STRCP1 pseudogene',
    'NCF1': 'NCF1B and NCF1C pseudogenes',
    'IKBKG': 'IKBKGP1 pseudogene',
    'HBA1': 'HBA2 paralogue',
    'HBA2': 'HBA1 paralogue',
    'OPN1LW': 'OPN1MW array',
    'NEB': 'triplicate repeat region',
    'CHEK2': 'CHEK2 pseudogenes',
}

# nomenclature
PREFERRED_TX = {
    'APC': 'NM_000038', 'ATP7B': 'NM_000053', 'BMPR1A': 'NM_004329',
    'BRCA1': 'NM_007294', 'BRCA2': 'NM_000059', 'PALB2': 'NM_024675',
    'MLH1': 'NM_000249', 'MSH2': 'NM_000251', 'MSH6': 'NM_000179',
    'MUTYH': 'NM_001128425', 'EPCAM': 'NM_002354', 'NF2': 'NM_000268',
    'PTEN': 'NM_000314', 'RB1': 'NM_000321', 'RET': 'NM_020975',
    'SDHAF2': 'NM_017841', 'SDHB': 'NM_003000', 'SDHC': 'NM_003001',
    'SDHD': 'NM_003002', 'SMAD4': 'NM_005359', 'STK11': 'NM_000455',
    'TP53': 'NM_000546', 'TSC1': 'NM_000368', 'TSC2': 'NM_000548',
    'VHL': 'NM_000551', 'WT1': 'NM_024426', 'MEN1': 'NM_130799',
    'MAX': 'NM_002382', 'TMEM127': 'NM_017849', 'CDH1': 'NM_004360',
    'ATM': 'NM_000051', 'BARD1': 'NM_000465', 'BRIP1': 'NM_032043',
    'RAD51C': 'NM_058216', 'RAD51D': 'NM_002878',
    'ACTA2': 'NM_001613', 'MYH11': 'NM_002474', 'COL3A1': 'NM_000090',
    'FBN1': 'NM_000138', 'SMAD3': 'NM_005902', 'TGFBR1': 'NM_004612',
    'TGFBR2': 'NM_003242', 'DSC2': 'NM_024422', 'DSG2': 'NM_001943',
    'DSP': 'NM_004415', 'PKP2': 'NM_004572', 'TMEM43': 'NM_024334',
    'BAG3': 'NM_004281', 'DES': 'NM_001927', 'FLNC': 'NM_001458',
    'LMNA': 'NM_170707', 'PLN': 'NM_002667', 'RBM20': 'NM_001134363',
    'SCN5A': 'NM_198056', 'TNNC1': 'NM_003280', 'TNNT2': 'NM_001001430',
    'TTN': 'NM_001267550', 'APOB': 'NM_000384', 'LDLR': 'NM_000527',
    'PCSK9': 'NM_174936', 'ACTC1': 'NM_005159', 'MYBPC3': 'NM_000256',
    'MYH7': 'NM_000257', 'MYL2': 'NM_000432', 'MYL3': 'NM_000258',
    'PRKAG2': 'NM_016203', 'TNNI3': 'NM_000363', 'TPM1': 'NM_001018005',
    'ACVRL1': 'NM_000020', 'ENG': 'NM_000118', 'CALM1': 'NM_006888',
    'CALM2': 'NM_001743', 'CALM3': 'NM_005184', 'CASQ2': 'NM_001232',
    'TRDN': 'NM_006073', 'KCNH2': 'NM_000238', 'KCNQ1': 'NM_000218',
    'RYR2': 'NM_001035', 'TTR': 'NM_000371', 'ABCD1': 'NM_000033',
    'BTD': 'NM_000060', 'CYP27A1': 'NM_000784', 'GAA': 'NM_000152',
    'GLA': 'NM_000169', 'HFE': 'NM_000410', 'OTC': 'NM_000531',
    'CACNA1S': 'NM_000069', 'RYR1': 'NM_000540', 'HNF1A': 'NM_000545',
    'RPE65': 'NM_000329', 'COL4A3': 'NM_000091', 'COL4A4': 'NM_000092',
    'COL4A5': 'NM_000495', 'UMOD': 'NM_003361', 'NPHS1': 'NM_004646',
    'NPHS2': 'NM_014625', 'PKD1': 'NM_001009944', 'PKD2': 'NM_000297',
    'APOL1': 'NM_003661', 'APP': 'NM_000484', 'PSEN1': 'NM_000021',
    'PSEN2': 'NM_000447', 'APOE': 'NM_000041', 'LRRK2': 'NM_198578',
    'SNCA': 'NM_000345', 'PINK1': 'NM_032409', 'GCK': 'NM_000162',
    'HNF4A': 'NM_175914', 'HNF1B': 'NM_000458', 'KCNJ11': 'NM_000525',
    'ABCC8': 'NM_000352', 'INS': 'NM_000207', 'APOA5': 'NM_052968',
    'LPL': 'NM_000237', 'ABCG5': 'NM_022436', 'ABCG8': 'NM_022437',
    'LIPA': 'NM_000235', 'PROC': 'NM_000312', 'PROS1': 'NM_000313',
    'SERPINC1': 'NM_000488', 'VWF': 'NM_000552', 'F2': 'NM_000506',
    'F5': 'NM_000130', 'F8': 'NM_000132', 'F9': 'NM_000133',
    'SLCO1B1': 'NM_006446', 'SERPINA1': 'NM_000295', 'TCF7L2': 'NM_030756',
    'PNPLA3': 'NM_025225',
}

AACHANGE_RE = re.compile(
    r'(?P<gene>[^:,]+):(?P<tx>N[MR]_\d+)(?:\.\d+)?:(?P<exon>[^:]*):'
    r'(?P<c>c\.[^:,]+)(?::(?P<p>p\.[^:,]+))?')

# curated HGVS
CURATED_HGVS = {
    'rs75030001':  ('NM_020975.6', 'c.2370G>T', 'p.Leu790Phe', 'G', 'T'),
    'rs1800562':   ('NM_000410.4', 'c.845G>A', 'p.Cys282Tyr', 'G', 'A'),
    'rs1799945':   ('NM_000410.4', 'c.187C>G', 'p.His63Asp', 'C', 'G'),
    'rs34612342':  ('NM_001128425.2', 'c.536A>G', 'p.Tyr179Cys', 'T', 'C'),
    'rs421016':    ('NM_000157.4', 'c.1448T>C', 'p.Leu483Pro', 'A', 'G'),
    'chr2:179418944': ('NM_001267550.2', 'c.88895-1G>A', '', 'C', 'T'),
    'rs6025':      ('NM_000130.5', 'c.1601G>A', 'p.Arg534Gln', 'T', 'C'),
    'rs1799963':   ('NM_000506.5', 'c.*97G>A', '', 'G', 'A'),
    'rs1801133':   ('NM_005957.5', 'c.665C>T', 'p.Ala222Val', 'G', 'A'),
    'rs1801131':   ('NM_005957.5', 'c.1286A>C', 'p.Glu429Ala', 'T', 'G'),
    'rs671':       ('NM_000690.4', 'c.1510G>A', 'p.Glu504Lys', 'G', 'A'),
    'rs1229984':   ('NM_000668.5', 'c.143A>G', 'p.His48Arg', 'T', 'C'),
    'rs1426654':   ('NM_205850.3', 'c.331G>A', 'p.Ala111Thr', 'A', 'G'),
    'rs16891982':  ('NM_016180.5', 'c.1122C>G', 'p.Phe374Leu', 'C', 'G'),
    'rs4149056':   ('NM_006446.5', 'c.521T>C', 'p.Val174Ala', 'T', 'C'),
    'rs2306283':   ('NM_006446.5', 'c.388A>G', 'p.Asn130Asp', 'A', 'G'),
    'rs116855232': ('NM_018283.4', 'c.415C>T', 'p.Arg139Cys', 'C', 'T'),
    'rs1142345':   ('NM_000367.5', 'c.719A>G', 'p.Tyr240Cys', 'T', 'C'),
    'rs1800460':   ('NM_000367.5', 'c.460G>A', 'p.Ala154Thr', 'C', 'T'),
    'rs1799853':   ('NM_000771.4', 'c.430C>T', 'p.Arg144Cys', 'C', 'T'),
    'rs1057910':   ('NM_000771.4', 'c.1075A>C', 'p.Ile359Leu', 'A', 'C'),
    'rs4244285':   ('NM_000769.4', 'c.681G>A', 'p.Pro227=', 'G', 'A'),
    'rs4986893':   ('NM_000769.4', 'c.636G>A', 'p.Trp212Ter', 'G', 'A'),
    'rs4148323':   ('NM_000463.3', 'c.211G>A', 'p.Gly71Arg', 'G', 'A'),
    'rs2231142':   ('NM_004827.3', 'c.421C>A', 'p.Gln141Lys', 'G', 'T'),
    'rs1799971':   ('NM_000914.5', 'c.118A>G', 'p.Asn40Asp', 'A', 'G'),
    'rs121918154': ('NM_000312.4', 'c.814C>T', 'p.Arg272Cys', 'C', 'T'),
    'chr2:179466384': ('NM_001267550.2', 'c.55432+1G>T', '', 'C', 'A'),
    'rs201079485': ('NM_001371904.1', 'c.289C>T', 'p.Gln97Ter', 'G', 'A'),
    'chr13:52515216': ('NM_000053.4', 'c.3556+1G>T', '', 'C', 'A'),
    'rs76151636':  ('NM_000053.4', 'c.3207C>A', 'p.His1069Gln', 'G', 'T'),
    'rs751710854': ('NM_000053.4', 'c.2383C>T', 'p.Leu795Phe', 'G', 'A'),
    'rs1162633293': ('NM_003361.4', 'c.314G>A', 'p.Gly105Asp', 'C', 'T'),
    'rs750621215': ('NM_002878.4', 'c.898C>T', 'p.Arg300Ter', 'G', 'A'),
    'rs137886232': ('NM_002878.4', 'c.757C>T', 'p.Arg253Ter', 'G', 'A'),
    'rs557344672': ('NM_000527.5', 'c.542C>T', 'p.Pro181Leu', 'C', 'T'),
    'rs730882109': ('NM_000527.5', 'c.1747C>T', 'p.His583Tyr', 'C', 'T'),
    'rs587778777': ('NM_000784.4', 'c.1184+1G>A', '', 'G', 'A'),
}


def parse_aachange(field):
    if not isinstance(field, str) or field in ('.', ''):
        return []
    return [m.groupdict() for m in AACHANGE_RE.finditer(field)]


def genes_of(gene_field):
    if not isinstance(gene_field, str):
        return set()
    return {g.strip() for g in gene_field.split(',') if g.strip()}


def gene_mask(frame, gene_set):
    return frame['Gene.refGene'].apply(lambda g: bool(genes_of(g) & gene_set))


def pick_transcript(field, gene_field):
    hits = parse_aachange(field)
    if not hits:
        return pd.Series({'Transcript': pd.NA, 'Exon': pd.NA, 'HGVS_c': pd.NA,
                          'HGVS_p': pd.NA, 'Transcript_is_preferred': pd.NA})
    wanted = {PREFERRED_TX[g] for g in genes_of(gene_field) if g in PREFERRED_TX}
    chosen = next((h for h in hits if h['tx'] in wanted), None)
    preferred = chosen is not None
    if chosen is None:
        chosen = sorted(hits, key=lambda h: h['tx'])[0]
    return pd.Series({'Transcript': chosen['tx'], 'Exon': chosen['exon'] or pd.NA,
                      'HGVS_c': chosen['c'], 'HGVS_p': chosen['p'] or pd.NA,
                      'Transcript_is_preferred': preferred})


def curated_hgvs_for(rsid, ref, alt, chrom=None, pos=None):
    keys = []
    if chrom is not None and pos is not None and pd.notna(pos):
        keys.append(f'{chrom}:{int(pos)}')
    if rsid is not None and str(rsid) not in ('.', 'nan', '<NA>', ''):
        keys.append(str(rsid))
    e = next((CURATED_HGVS[k] for k in keys if k in CURATED_HGVS), None)
    if e is None:
        return (None, None, None, 'no entry')
    tx, c, p_, e_ref, e_alt = e
    if str(ref) != e_ref or str(alt) != e_alt:
        return (None, None, None, f'entry is for {e_ref}>{e_alt}, this row is {ref}>{alt}')
    return (tx, c, p_, 'match')

# inheritance and ACMG SF v3.3
INHERITANCE = {
    'AR': ['ATP7B', 'MUTYH', 'HFE', 'CYP27A1', 'GAA', 'BTD', 'CASQ2', 'TRDN',
           'COL4A3', 'COL4A4', 'NPHS1', 'NPHS2', 'PKHD1', 'HJV', 'HAMP', 'TFR2',
           'SLC40A1', 'LDLRAP1', 'ABCG5', 'ABCG8', 'LIPA', 'SERPINA1', 'PINK1',
           'PARK2', 'RPE65', 'MBOAT7'],
    'X-linked': ['GLA', 'OTC', 'ABCD1', 'G6PD', 'COL4A5', 'F8', 'F9'],
    'AD or AR': ['VWF', 'PROC', 'PROS1', 'RYR1', 'MYH7', 'SCN5A', 'KCNQ1', 'KCNH2',
                 'ABCC8', 'KCNJ11', 'LPL'],
}
INHERITANCE_SPECIAL = {
    'APOA5': 'complex; heterozygotes are risk alleles',
    'APOE': 'risk allele',
    'TCF7L2': 'common risk allele',
    'SLCO1B1': 'drug-response locus',
}


def inheritance(gene_field):
    g = genes_of(gene_field)
    special = [INHERITANCE_SPECIAL[x] for x in sorted(g) if x in INHERITANCE_SPECIAL]
    if special:
        return special[0]
    for mode, label in (('X-linked', 'X-linked'),
                        ('AR', 'AR (two variants needed)'),
                        ('AD or AR', 'AD or AR, depends on the phenotype')):
        if g & set(INHERITANCE[mode]):
            return label
    return 'AD (heterozygous)'


ACMG_SF_V33 = {
    'APC', 'BMPR1A', 'BRCA1', 'BRCA2', 'PALB2', 'MAX', 'TMEM127', 'MEN1',
    'MLH1', 'MSH2', 'MSH6', 'PMS2', 'MUTYH', 'NF2', 'PTEN', 'RB1', 'RET',
    'SDHAF2', 'SDHB', 'SDHC', 'SDHD', 'SMAD4', 'STK11', 'TP53', 'TSC1', 'TSC2',
    'VHL', 'WT1',
    'ACTA2', 'MYH11', 'COL3A1', 'FBN1', 'SMAD3', 'TGFBR1', 'TGFBR2',
    'DSC2', 'DSG2', 'DSP', 'PKP2', 'TMEM43',
    'BAG3', 'DES', 'FLNC', 'LMNA', 'PLN', 'RBM20', 'SCN5A', 'TNNC1', 'TNNT2', 'TTN',
    'APOB', 'LDLR', 'PCSK9',
    'ACTC1', 'MYBPC3', 'MYH7', 'MYL2', 'MYL3', 'PRKAG2', 'TNNI3', 'TPM1',
    'ACVRL1', 'ENG',
    'CALM1', 'CALM2', 'CALM3', 'CASQ2', 'TRDN', 'KCNH2', 'KCNQ1', 'RYR2', 'TTR',
    'ABCD1', 'BTD', 'CYP27A1', 'GAA', 'GLA', 'OTC',
    'ATP7B', 'CACNA1S', 'HFE', 'HNF1A', 'RPE65', 'RYR1',
}
assert len(ACMG_SF_V33) == 84, f'ACMG SF v3.3 list holds {len(ACMG_SF_V33)} genes, expected 84'
ACMG_V33_NEW = {'PLN', 'ABCD1', 'CYP27A1'}

ACMG_FOOTNOTE = {
    **{g: 'reported only when two P/LP variants are identified'
       for g in ['MUTYH', 'CASQ2', 'TRDN', 'BTD', 'CYP27A1', 'GAA', 'ATP7B',
                 'HNF1A', 'RPE65']},
    'HFE': 'reported only for p.C282Y homozygotes',
    **{g: 'reported in hemizygous, homozygous or compound heterozygous state'
       for g in ['ABCD1', 'GLA', 'OTC']},
    **{g: 'reported only for malignant hyperthermia susceptibility'
       for g in ['RYR1', 'CACNA1S']},
    'TTN': 'reported only for truncating variants in high-PSI cardiac exons',
    'RET': 'gene-disease pair is MEN2 and familial medullary thyroid carcinoma',
    'MYH7': 'gene-disease pair is cardiomyopathy',
    'SCN5A': 'gene-disease pair is long QT and Brugada syndrome',
    'TTR': 'gene-disease pair is hereditary transthyretin amyloidosis',
}

EFFECT_ALLELE_IS_REF = {'rs6025', 'rs1229984', 'rs1042725', 'rs776746', 'rs1426654',
                        'rs1208'}
REFERENCE_NOTE = {
    'rs6025': 'GRCh37 carries the Factor V Leiden allele',
    'rs1426654': 'GRCh37 carries the derived 111Thr allele',
    'rs776746': 'GRCh37 carries the CYP3A5*3 allele',
    'rs1229984': 'GRCh37 carries the ADH1B*2 allele',
    'rs1042725': 'GRCh37 carries the height-increasing allele',
    'rs1208': 'GRCh37 carries NAT2 803G',
}
COMPLEMENT = {'A': 'T', 'T': 'A', 'C': 'G', 'G': 'C', '-': '-'}

# statistics
def wilson_ci(k, n, z=1.96):
    if n is None or n == 0 or pd.isna(n):
        return (np.nan, np.nan)
    k, n = float(k), float(n)
    p = k / n
    d = 1 + z ** 2 / n
    centre = (p + z ** 2 / (2 * n)) / d
    half = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / d
    return (max(0.0, centre - half), min(1.0, centre + half))


def one_in_n(k, n, upper=None):
    if k:
        return f'1 in {int(round(n / k))}'
    if upper and upper > 0:
        return f'0 observed, upper bound 1 in {int(round(1 / upper))}'
    return '0 observed'

FIG_W2 = 180
FIG_W1 = 88
MM = 1 / 25.4
W1, W15, W2 = 88 * MM, 120 * MM, 180 * MM
FIG_W1, FIG_W2 = 88, 180
DPI_RASTER = 1200
FIG_FORMATS = ['tiff', 'png']
SMALLEST_PT = 8.0
FS, FS_PANEL, FS_MAX = 8, 10, 12
TEXT_DARK = '#4A4A4A'

MAIN_FIG = {'pgx': 4, 'plp': 5}
SUPP_NUM = {'fig_population_context': 7,
            'tab_trait': 18, 'tab_pgx': 19, 'tab_pgx_absent': 20,
            'tab_excluded': 21, 'tab_hwe': 22, 'tab_acmg': 23,
            'data': 4}
SOURCE = {}                    
ROW_MM = 4.6

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'Liberation Sans', 'DejaVu Sans'],
    'font.size': 8, 'axes.labelsize': 8, 'axes.titlesize': 8,
    'xtick.labelsize': 8, 'ytick.labelsize': 8, 'legend.fontsize': 8,
    'legend.title_fontsize': 8, 'hatch.linewidth': 0.5,
    'axes.linewidth': 0.6, 'xtick.major.width': 0.6, 'ytick.major.width': 0.6,
    'xtick.major.size': 2, 'ytick.major.size': 2,
    'axes.edgecolor': '#3C3C3C', 'xtick.color': '#3C3C3C', 'ytick.color': '#3C3C3C',
    'text.color': '#1A1A1A', 'axes.labelcolor': '#1A1A1A',
    'axes.spines.top': False, 'axes.spines.right': False,
    'legend.frameon': False, 'legend.handletextpad': 0.4, 'legend.borderpad': 0.2,
    'grid.color': '#C8C8C8', 'grid.linewidth': 0.4, 'grid.linestyle': '-',
    'lines.linewidth': 0.8,
    'pdf.fonttype': 42, 'ps.fonttype': 42, 'svg.fonttype': 'none',
    'figure.facecolor': 'white', 'savefig.facecolor': 'white',
    'figure.dpi': 120, 'savefig.dpi': DPI_RASTER,
    'savefig.bbox': None, 'savefig.pad_inches': 0.0,
})

INK, RULE, FAINT, BAND = '#1A1A1A', '#3C3C3C', '#B0B0B0', '#4A4A4A'

KZ, KZ_SOFT = '#22303C', '#4A6376'     # the cohort, and any set this paper reports
EAST = '#2C6E9B'      # East Eurasian; gnomAD EAS; 1KG EAS
WEST = '#B5502F'      # West Eurasian; gnomAD NFE; 1KG EUR; gnomAD FIN
SOUTH = '#4E8C86'     # 1KG SAS; gnomAD SAS
AFRICA = '#8A6BA8'    # gnomAD AFR
GLOBAL = '#7F8C97'    # gnomAD ALL, GME, populations pooled
UNRES = '#A0A0A0'     # unresolved, not assigned, no call
ACCENT = '#C8912A'    # one-off highlight: a threshold, a founder cluster
EXCLUDED = '#FFFFFF'  # a subset removed by a stated rule, drawn as an open bar

SET_BG, SET_FG = '#AEBECB', KZ         # a set against a subset of itself
SLATE = ['#22303C', '#3A4E60', '#6E8598', '#AEBECB', '#DCE4EA']

PANEL_COLOUR = {
    'gnomAD_EAS': EAST, '1KG_EAS': EAST, 'EAS': EAST,
    'gnomAD_NFE': WEST, '1KG_EUR': WEST, 'gnomAD_FIN': WEST, 'EUR': WEST, 'NFE': WEST,
    '1KG_SAS': SOUTH, 'gnomAD_SAS': SOUTH, 'SAS': SOUTH,
    'gnomAD_AFR': AFRICA, 'AFR': AFRICA,
    'gnomAD_ALL': GLOBAL, 'GME_ALL': GLOBAL, 'ALL': GLOBAL,
}
ANC_ORDER = ['East', 'West', 'Unassigned']
ANC_COLORS = {'East': EAST, 'West': WEST, 'Unassigned': UNRES}
ANC_LABEL = {'East': 'East Eurasian', 'West': 'West Eurasian',
             'Unassigned': 'Unresolved'}

BAR_EDGE = {'edgecolor': RULE, 'linewidth': 0.3}
OPEN_BAR = {'facecolor': EXCLUDED, 'edgecolor': KZ, 'linewidth': 0.5}

HEAT = mcolors.LinearSegmentedColormap.from_list(
    'kz_slate', ['#FFFFFF', '#DCE4EA', '#AEBECB', '#6E8598', '#3A4E60', '#1B2A36'],
    N=256)

COL = {'cohort': KZ, 'eur': WEST, 'eas': EAST, 'sas': SOUTH, 'accent': ACCENT,
       'sky': SLATE[2], 'dark': RULE, 'mid': SLATE[2], 'light': '#C3CDD6'}


def on(hex_color):
    """Ink colour that stays legible on the given fill."""
    r, g, b = mcolors.to_rgb(hex_color)
    return '#FFFFFF' if 0.299 * r + 0.587 * g + 0.114 * b < 0.55 else INK


def clade_palette(clades, anc_of):
    """A ramp per ancestry, so a stacked bar of many clades stays readable."""
    ramps = {'East': ('#12405F', '#C3D8E5'), 'West': ('#7A3418', '#EDCBB8'),
             'Unassigned': ('#8A8A8A', '#D8D8D8')}
    out = {}
    for a in ANC_ORDER:
        sub = [c for c in clades if anc_of.get(c) == a]
        if not sub:
            continue
        cm = mcolors.LinearSegmentedColormap.from_list('ramp', list(ramps[a]))
        for i, c in enumerate(sub):
            out[c] = mcolors.to_hex(cm(i / max(1, len(sub) - 1)))
    return out


def panel(ax, letter, dx=-26, dy=6):
    """Panel letter at a fixed offset in points, identical on panels of any size."""
    ax.annotate(letter, xy=(0, 1), xycoords='axes fraction',
                xytext=(dx, dy), textcoords='offset points',
                fontsize=FS_PANEL, fontweight='bold', va='bottom', ha='left', color=INK)


panel_label = panel


def despine(ax, keep=('left', 'bottom')):
    for name, sp in ax.spines.items():
        sp.set_visible(name in keep)


def check_type_size(fig, floor=SMALLEST_PT):
    """Raise any text below the 8 pt floor (cap above 12 pt) and report what changed."""
    changed = []
    for t in fig.findobj(mpl.text.Text):
        if not t.get_visible() or not t.get_text().strip():
            continue
        s = t.get_fontsize()
        if s < floor - 1e-6 or s > FS_MAX + 1e-6:
            changed.append((t.get_text()[:24], round(s, 2)))
            t.set_fontsize(min(max(s, floor), FS_MAX))
    if changed:
        print(f'  NOTE: {len(changed)} text item(s) raised to {floor:g} pt, check the layout: '
              f'{changed[:6]}')
    return changed


def save_figure(fig, stem, outdir=None):
    """PNG (RGB) and TIFF (CMYK, LZW) at 1200 dpi, named with the figure number."""
    from PIL import Image
    check_type_size(fig)
    outdir = OUT_DIR if outdir is None else outdir
    os.makedirs(outdir, exist_ok=True)
    png = os.path.join(outdir, f'{stem}.png')
    fig.savefig(png, format='png', dpi=DPI_RASTER, bbox_inches='tight', pad_inches=0.03)
    paths = []
    if 'tiff' in FIG_FORMATS:
        tif = os.path.join(outdir, f'{stem}.tiff')
        with Image.open(png) as im:
            im.convert('RGB').convert('CMYK').save(tif, compression='tiff_lzw',
                                                   dpi=(DPI_RASTER, DPI_RASTER))
        paths.append(tif)
    if 'png' in FIG_FORMATS:
        paths.append(png)
    else:
        os.remove(png)
    for p in paths:
        with Image.open(p) as im:
            w, h, mode = *im.size, im.mode
        wmm = w / DPI_RASTER * 25.4
        print(f'saved {os.path.basename(p)}: {mode}, {wmm:.0f} x {h / DPI_RASTER * 25.4:.0f} mm '
              f'at {DPI_RASTER} dpi, {os.path.getsize(p) / 1e6:.1f} MB'
              + ('   WARNING: wider than 183 mm' if wmm > 183 else ''))
    return paths


save_fig = save_figure

CONSEQ_COLOUR = {'stopgain': SLATE[0], 'frameshift deletion': '#3A4E60',
                 'frameshift insertion': KZ_SOFT, 'nonsynonymous SNV': SLATE[2],
                 'nonframeshift deletion': SLATE[3],
                 'nonframeshift insertion': SLATE[4],
                 'stoploss': ACCENT, 'startloss': ACCENT, '.': UNRES}
CONSEQ_LABEL = {'.': 'splicing or unannotated'}


def conseq_colour(v):
    return CONSEQ_COLOUR.get(str(v), UNRES)


def conseq_label(v):
    return CONSEQ_LABEL.get(str(v), str(v).replace(' SNV', ''))



def wilson(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan, np.nan)
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z / d * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return p, max(0.0, c - h), min(1.0, c + h)


print(f'{COHORT}, n = {N_COHORT} ({N_FEMALE} female, {N_MALE} male), {N_CHR_KZ} chromosomes')
print(f'{CALLER_SAMPLE} per sample, {CALLER_JOINT} joint genotyping, {ANNOVAR_BUILD}, '
      f'{CLINVAR_RELEASE}')
print(f'reported ClinVar review status: >= {MIN_CLINVAR_STARS} stars')
print(f'preferred transcripts {len(PREFERRED_TX)}, curated HGVS {len(CURATED_HGVS)}, '
      f'repeat tracts {len(REPEAT_TRACTS_HG19)}, paralogue genes {len(PARALOGUE_GENES)}')

## 2. Loading and genotype counts

The table from notebook 02 is cut to the required columns, read in chunks, filtered to rows
carrying an alternate allele or a ClinVar pathogenic assertion, and cached as Parquet.

Genotypes are parsed column-wise: the `FORMAT` suffix is dropped, phasing is normalised, a
genotype with a missing allele is a no-call, a haploid call contributes one chromosome, and the
allele index is matched to the VCF `ALT` list at multiallelic sites. Counts are reconciled
against `AC_pass`, `AN_pass` and `n_hom_alt_pass` from notebook 02 and the comparison is
recorded in `QC_Concordance`.

In [ ]:
KEEP_COLS = {
    'Chr', 'Start', 'End', 'Ref', 'Alt',
    'Func.refGene', 'Gene.refGene', 'ExonicFunc.refGene', 'AAChange.refGene', 'avsnp151',
    'SIFT_pred', 'Polyphen2_HVAR_pred',
    '1000G_ALL', '1000G_EAS', '1000G_EUR', '1000G_SAS',
    'AF', 'AF_popmax', 'AF_afr', 'AF_sas', 'AF_eas', 'AF_nfe',
    'AF.1', 'AF_popmax.1', 'AF_afr.1', 'AF_sas.1', 'AF_eas.1', 'AF_nfe.1',
    'CLNSIG', 'CLNDN', 'CLNREVSTAT',
    '#CHROM', 'POS', 'REF', 'ALT', 'QUAL', 'FILTER',
    'AC_all', 'AN_all', 'AF_all', 'MAF_all',
    'AC_pass', 'AN_pass', 'AF_pass', 'MAF_pass', 'n_hom_alt_pass', 'hom_alt_freq_pass',
}
GT_RE = re.compile(r'^\d{3}_')

NUM_F32 = ['AF_pass', 'MAF_pass', 'AF_all', 'MAF_all', 'hom_alt_freq_pass',
           'AC_pass', 'AN_pass', 'AC_all', 'AN_all', 'n_hom_alt_pass',
           '1000G_ALL', '1000G_EAS', '1000G_EUR', '1000G_SAS',
           'AF', 'AF_eas', 'AF_nfe', 'AF_sas', 'AF_afr', 'AF_popmax',
           'AF.1', 'AF_eas.1', 'AF_nfe.1', 'AF_sas.1', 'AF_afr.1', 'AF_popmax.1']
NUM_F64 = ['POS', 'Start', 'End', 'QUAL']
STR_COLS = ['Chr', '#CHROM', 'Ref', 'Alt', 'REF', 'ALT', 'FILTER',
            'Func.refGene', 'Gene.refGene', 'ExonicFunc.refGene', 'AAChange.refGene',
            'avsnp151', 'SIFT_pred', 'Polyphen2_HVAR_pred',
            'CLNSIG', 'CLNDN', 'CLNREVSTAT']


def harmonise_dtypes(frame, gt_columns, verbose=True):
    n_num = n_str = 0
    gt_set = set(gt_columns)
    for c in NUM_F64:
        if c in frame.columns and frame[c].dtype != 'float64':
            frame[c] = pd.to_numeric(frame[c], errors='coerce').astype('float64')
            n_num += 1
    for c in set(NUM_F32) & set(frame.columns):
        if frame[c].dtype != 'float32':
            frame[c] = pd.to_numeric(frame[c], errors='coerce').astype('float32')
            n_num += 1
    for c in frame.columns:
        if c in gt_set or c in NUM_F64 or c in NUM_F32:
            continue
        if frame[c].dtype == object:
            frame[c] = frame[c].astype('string')
            n_str += 1
    if verbose:
        print(f'  harmonised {n_num} numeric, {n_str} string columns')
    return frame


def allele_index(frame, quiet=False):
    if 'ALT' not in frame.columns:
        return pd.Series(1.0, index=frame.index)
    alt_vcf = frame['ALT'].astype('string').fillna('')
    multi = alt_vcf.str.contains(',', na=False)
    idx = pd.Series(1.0, index=frame.index)
    if not multi.any():
        return idx
    ann = frame['Alt'].astype('string').fillna('')
    for i in frame.index[multi]:
        alts = [a.strip() for a in str(alt_vcf.loc[i]).split(',')]
        a = str(ann.loc[i])
        idx.loc[i] = float(alts.index(a) + 1) if a in alts else np.nan
    if not quiet:
        print(f'  multiallelic rows {int(multi.sum()):,}, '
              f'allele index unresolved {int(idx.isna().sum()):,}')
    return idx


def genotype_matrix(frame, gt_columns, quiet=False):
    idx = allele_index(frame, quiet=quiet)
    dosage, ploidy = {}, {}
    n_haploid = 0
    for c in tqdm(gt_columns, desc='genotypes', leave=False):
        g = (frame[c].astype('string').str.split(':').str[0]
             .str.replace('|', '/', regex=False))
        parts = g.str.split('/', n=1, expand=True)
        if parts.shape[1] == 1:
            parts[1] = pd.NA
        a1 = pd.to_numeric(parts[0], errors='coerce')
        a2 = pd.to_numeric(parts[1], errors='coerce')

        def _match(a):
            return (a.eq(idx).where(idx.notna(), a.gt(0)).fillna(False).astype('int8'))

        d = _match(a1) + _match(a2)
        p = a1.notna().astype('int8') + a2.notna().astype('int8')
        n_haploid += int((p == 1).sum())
        dosage[c] = d.where(p > 0, 0).astype('int8')
        ploidy[c] = p
    if n_haploid and not quiet:
        print(f'  haploid calls counted as one chromosome: {n_haploid:,}')
    return (pd.DataFrame(dosage, index=frame.index),
            pd.DataFrame(ploidy, index=frame.index))


def add_counts(frame, gt_columns, label='', quiet=False):
    _idx = allele_index(frame, quiet=True)
    dos, plo = genotype_matrix(frame, gt_columns, quiet=quiet)
    called = plo > 0
    out = frame.copy()
    out['Allele_index_unresolved'] = _idx.isna().values
    out['n_called'] = called.sum(axis=1).astype(int)
    out['n_hom_alt'] = ((dos == 2) & (plo == 2)).sum(axis=1).astype(int)
    out['n_het'] = ((dos == 1) & (plo == 2)).sum(axis=1).astype(int)
    out['n_hemi_alt'] = ((dos == 1) & (plo == 1)).sum(axis=1).astype(int)
    out['n_hom_ref'] = (out['n_called'] - out['n_het'] - out['n_hom_alt']
                        - out['n_hemi_alt']).astype(int)
    out['n_carriers'] = (dos > 0).sum(axis=1).astype(int)
    out['AC_gt'] = dos.sum(axis=1).astype(int)
    out['AN_gt'] = plo.sum(axis=1).astype(int)
    out['AF_gt'] = np.where(out['AN_gt'] > 0, out['AC_gt'] / out['AN_gt'], np.nan)
    out['genotyping_rate'] = out['n_called'] / N_COHORT
    out['Low_call_rate'] = out['genotyping_rate'] < MIN_CALL_RATE
    if 'ALT' in out.columns:
        alt_list = out['ALT'].astype('string').fillna('')
        out['VCF_multiallelic'] = alt_list.str.contains(',', na=False)
    else:
        out['VCF_multiallelic'] = False

    out['n_hom_or_hemi'] = out['n_hom_alt'] + out['n_hemi_alt']
    checks, bad_idx = [], pd.Index([])
    for gt_col, up_col in [('AC_gt', 'AC_pass'), ('AN_gt', 'AN_pass'),
                           ('n_hom_or_hemi', 'n_hom_alt_pass')]:
        if up_col not in out.columns:
            continue
        up = pd.to_numeric(out[up_col], errors='coerce')
        diff = out[gt_col] - up
        broken = out.index[(diff < 0).fillna(False)]
        bad_idx = bad_idx.union(broken)
        checks.append({'Field': f'{gt_col} vs {up_col}',
                       'Rows compared': int(up.notna().sum()),
                       'Upstream filtered some genotypes': int((diff > 0).fillna(False).sum()),
                       'Inconsistent rows': len(broken)})
    qc = pd.DataFrame(checks)
    out['Count_discordant'] = out.index.isin(bad_idx)

    if not quiet and label:
        print(f'  {label}: {len(out)} rows, {int(out["n_carriers"].sum())} carrier genotypes, '
              f'{int(out["Count_discordant"].sum())} inconsistent')
        if not qc.empty:
            display(qc)
    return out, dos, plo, qc


def effect_dosage(frame, dos, plo):
    is_ref = frame['avsnp151'].astype('string').isin(EFFECT_ALLELE_IS_REF)
    inv = is_ref.reindex(frame.index).fillna(False).values
    eff = dos.copy()
    eff.loc[inv, :] = (plo.loc[inv, :] - dos.loc[inv, :])
    return eff.where(plo > 0, other=pd.NA)

SLIM_MANIFEST = SLIM_FILE + '.done.json'
FORCE_RESLIM = False


def _slim_is_complete():
    if FORCE_RESLIM or not (os.path.exists(SLIM_FILE) and os.path.exists(SLIM_MANIFEST)):
        if os.path.exists(SLIM_FILE) and not os.path.exists(SLIM_MANIFEST):
            print('slim file present but has no manifest, treating it as an '
                  'interrupted run and rebuilding')
        return False
    try:
        info = json.load(open(SLIM_MANIFEST))
    except Exception:
        return False
    st = os.stat(INPUT_FILE)
    if info.get('source_size') != st.st_size:
        print('slim file was cut from a different input, rebuilding')
        return False
    if info.get('bytes') != os.path.getsize(SLIM_FILE):
        print('slim file changed size since it was written, rebuilding')
        return False
    return True


assert os.path.exists(INPUT_FILE), f'input not found: {INPUT_FILE}'
header = pd.read_csv(INPUT_FILE, sep='\t', nrows=0).columns
wanted = [c for c in header if c in KEEP_COLS or GT_RE.match(c)]

if len(wanted) == len(header):
    SLIM_FILE = INPUT_FILE
    print(f'input is already slim, {len(header)} columns, cut skipped')
elif _slim_is_complete():
    print(f'slim file reused, {os.path.getsize(SLIM_FILE)/1024**3:.1f} GB')
else:
    keep_idx = [i + 1 for i, c in enumerate(header) if c in KEEP_COLS or GT_RE.match(c)]
    print(f'raw columns {len(header)}, keeping {len(keep_idx)}')
    if not shutil.which('cut'):
        raise RuntimeError('cut not found on PATH')
    tmp = SLIM_FILE + '.partial'
    with open(tmp, 'w') as out:
        subprocess.run(['cut', '-f', ','.join(map(str, keep_idx)), INPUT_FILE],
                       stdout=out, check=True)
    os.replace(tmp, SLIM_FILE)      # only a finished cut ever appears under the real name
    with open(SLIM_MANIFEST, 'w') as fh:
        json.dump({'source': os.path.abspath(INPUT_FILE),
                   'source_size': os.stat(INPUT_FILE).st_size,
                   'bytes': os.path.getsize(SLIM_FILE),
                   'columns': len(keep_idx),
                   'written': datetime.now().isoformat(timespec='seconds')}, fh, indent=1)
    print(f'wrote slim file, {os.path.getsize(SLIM_FILE)/1024**3:.1f} GB')

df_raw, gt_cols = None, None
try:
    df_raw = pd.read_parquet(CKPT_FILE)
    gt_cols = [c for c in df_raw.columns if GT_RE.match(c)]
    print(f'loaded cache: {df_raw.shape[0]:,} rows')
except Exception:
    df_raw = None

if df_raw is None:
    header = pd.read_csv(SLIM_FILE, sep='\t', nrows=0).columns
    gt_cols = [c for c in header if GT_RE.match(c)]
    read_dtypes = {c: 'string' for c in list(gt_cols) + STR_COLS + NUM_F64 + NUM_F32
                   if c in header}
    keep = []
    for chunk in tqdm(pd.read_csv(SLIM_FILE, sep='\t', chunksize=400_000,
                                  dtype=read_dtypes, low_memory=False),
                      desc='loading', unit=' chunk'):
        observed = pd.to_numeric(chunk[AF_COL], errors='coerce') > 0
        plp = chunk.get('CLNSIG', pd.Series('', index=chunk.index)).fillna('').map(is_plp)
        sub = chunk[observed | plp]
        if not sub.empty:
            keep.append(sub)
    df_raw = pd.concat(keep, ignore_index=True)
    del keep
    gc.collect()
    for c in tqdm([c for c in gt_cols if c in df_raw.columns], desc='to category'):
        df_raw[c] = df_raw[c].astype('category')
    df_raw = harmonise_dtypes(df_raw, gt_cols)
    gc.collect()
    if not os.path.exists(CKPT_FILE):
        try:
            _tmp = CKPT_FILE + '.partial'
            df_raw.to_parquet(_tmp, compression='zstd')
            os.replace(_tmp, CKPT_FILE)   # an interrupted write leaves no cache to trust
        except Exception:
            pass

assert len(gt_cols) == N_COHORT, f'{len(gt_cols)} genotype columns, expected {N_COHORT}'

filter_col = next((c for c in ['FILTER', 'Filter', 'Otherinfo4', 'Otherinfo5']
                   if c in df_raw.columns), None)
is_pass = (df_raw[filter_col].astype(str).str.upper() == 'PASS' if filter_col
           else pd.Series(True, index=df_raw.index))

print(f'loaded {len(df_raw):,} rows, {int(is_pass.sum()):,} PASS, '
      f'{df_raw.memory_usage(deep=True).sum()/1e9:.2f} GB')

_probe = df_raw[pd.to_numeric(df_raw[AF_COL], errors='coerce') > 0].head(300)
if not _probe.empty:
    _p = add_counts(_probe, gt_cols, quiet=True)[0]
    assert _p['n_carriers'].sum() > 0, 'genotype parser produced no carriers at AF > 0'
    del _p
    gc.collect()

## 3. Pathogenic and likely pathogenic variants

A variant enters the tier 1 set when it is exonic or splicing, site-level `PASS`, classified
pathogenic or likely pathogenic in ClinVar without a conflicting or benign token, and observed
in the cohort. Repeat tracts and paralogue-bearing genes are removed before any counting.

The reported set adds a ClinVar review status of at least `MIN_CLINVAR_STARS` stars. Unfiltered
counts are retained as a sensitivity analysis in the `*_all_stars` sheets.

A carrier is an individual with at least one alternate allele at at least one variant in the
set. Intervals are Wilson score intervals on k / 219.

In [ ]:
exonic = df_raw['Func.refGene'].astype('string').str.contains('exonic|splicing',
                                                              case=False, na=False)
plp_mask = df_raw['CLNSIG'].astype('string').fillna('').map(is_plp)
observed = pd.to_numeric(df_raw[AF_COL], errors='coerce').fillna(0) > 0

df_plp = df_raw[exonic & plp_mask & is_pass & observed].copy()
print(f'exonic or splicing {int(exonic.sum()):,}, ClinVar P/LP, PASS and observed {len(df_plp):,}')


def in_repeat_tract(frame):
    chrom = frame['Chr'].astype('string').str.replace('^chr', '', regex=True)
    pos = pd.to_numeric(frame['Start'], errors='coerce')
    mask = pd.Series(False, index=frame.index)
    label = pd.Series(pd.NA, index=frame.index, dtype='string')
    for gene, c, lo, hi in REPEAT_TRACTS_HG19:
        hit = chrom.eq(c.replace('chr', '')) & pos.between(lo, hi)
        mask |= hit
        label = label.mask(hit, f'{gene} repeat tract, not sizeable from short reads')
    return mask, label


def in_paralogue_gene(frame):
    mask = gene_mask(frame, set(PARALOGUE_GENES))
    label = frame['Gene.refGene'].apply(
        lambda gf: '; '.join(f'{g}, {PARALOGUE_GENES[g]}'
                             for g in sorted(genes_of(gf) & set(PARALOGUE_GENES)))
        or pd.NA)
    return mask, label


excluded = []
for fn, tag in ((in_repeat_tract, 'repeat tract'), (in_paralogue_gene, 'paralogue')):
    m, lab = fn(df_plp)
    if m.any():
        sub = df_plp[m].copy()
        sub['Exclusion_reason'] = lab[m]
        excluded.append(sub)
        df_plp = df_plp[~m].copy()
    print(f'excluded, {tag}: {int(m.sum())}')

df_plp, dos_plp, plo_plp, qc_plp = add_counts(df_plp, gt_cols, label='tier 1 P/LP')

df_plp['Inheritance'] = df_plp['Gene.refGene'].apply(inheritance)
df_plp['ClinVar_stars'] = (df_plp.get('CLNREVSTAT', pd.Series(pd.NA, index=df_plp.index))
                           .astype('string').map(clinvar_stars))
df_plp['ClinVar_side_assertion'] = df_plp['CLNSIG'].astype('string').map(has_side_assertion)
df_plp['Reference_note'] = df_plp['avsnp151'].astype('string').map(REFERENCE_NOTE)
df_plp = pd.concat(
    [df_plp, df_plp.apply(lambda r: pick_transcript(r.get('AAChange.refGene'),
                                                    r.get('Gene.refGene')), axis=1)], axis=1)
df_plp['Transcript_is_preferred'] = df_plp['Transcript_is_preferred'].astype('boolean')


def resolve_hgvs(row):
    tx, c, p_, verdict = curated_hgvs_for(row.get('avsnp151'), row.get('Ref'),
                                          row.get('Alt'), row.get('Chr'), row.get('Start'))
    if verdict == 'match':
        return pd.Series({'Transcript': tx, 'HGVS_c': c, 'HGVS_p': p_ or pd.NA,
                          'HGVS_source': 'curated', 'HGVS_requires_curation': False,
                          'HGVS_annovar_unverified': pd.NA})
    if verdict != 'no entry':
        return pd.Series({'Transcript': pd.NA, 'HGVS_c': pd.NA, 'HGVS_p': pd.NA,
                          'HGVS_source': f'curated entry rejected: {verdict}',
                          'HGVS_requires_curation': True,
                          'HGVS_annovar_unverified': row.get('AAChange.refGene')})
    if row.get('Transcript_is_preferred') is True:
        return pd.Series({'Transcript': row.get('Transcript'), 'HGVS_c': row.get('HGVS_c'),
                          'HGVS_p': row.get('HGVS_p'),
                          'HGVS_source': 'preferred refGene transcript',
                          'HGVS_requires_curation': False,
                          'HGVS_annovar_unverified': pd.NA})
    raw = row.get('AAChange.refGene')
    return pd.Series({'Transcript': pd.NA, 'HGVS_c': pd.NA, 'HGVS_p': pd.NA,
                      'HGVS_source': 'not resolved', 'HGVS_requires_curation': True,
                      'HGVS_annovar_unverified': raw if isinstance(raw, str) else pd.NA})


_res = df_plp.apply(resolve_hgvs, axis=1)
for c in _res.columns:
    df_plp[c] = _res[c]
df_plp['HGVS_requires_curation'] = df_plp['HGVS_requires_curation'].astype('boolean')

df_excluded = (pd.concat(excluded, ignore_index=True) if excluded
               else pd.DataFrame(columns=['Exclusion_reason']))

stars_all = pd.to_numeric(df_plp['ClinVar_stars'], errors='coerce')
df_rep = df_plp[stars_all.ge(MIN_CLINVAR_STARS).fillna(False)].copy()
n_plp_all = len(df_plp)
n_plp_reported = len(df_rep)
print(f'tier 1 P/LP variants: {n_plp_all}; reported at >= {MIN_CLINVAR_STARS} stars: '
      f'{n_plp_reported}')

## 4. Disease panels and carrier frequencies

Panels are gene sets, not clinical assays. Panels that mix monogenic genes with common-variant
risk loci are labelled in `Panel composition`; genes appear in more than one panel, so panel
carrier frequencies are not independent and are not summed.

In [ ]:
PANELS = {
    'Cardiovascular - Familial Hypercholesterolaemia': [
        'LDLR', 'APOB', 'PCSK9', 'LDLRAP1', 'STAP1'],
    'Cardiovascular - Hypertrophic Cardiomyopathy': [
        'MYH7', 'MYBPC3', 'TNNT2', 'TNNI3', 'TPM1', 'MYL2', 'MYL3', 'ACTC1', 'TNNC1', 'PLN'],
    'Cardiovascular - Dilated Cardiomyopathy': [
        'TTN', 'LMNA', 'SCN5A', 'MYH7', 'MYBPC3', 'RBM20', 'FLNC', 'DSP', 'BAG3', 'NEXN'],
    'Cardiovascular - Arrhythmia / Channelopathy': [
        'SCN5A', 'KCNQ1', 'KCNH2', 'KCNE1', 'KCNE2', 'RYR2', 'CASQ2', 'ANK2', 'HCN4',
        'CALM1', 'CALM2', 'CALM3'],
    'Cardiovascular - Hypertension': [
        'AGT', 'AGTR1', 'ACE', 'REN', 'NR3C2', 'HSD11B2', 'SCNN1B', 'SCNN1G',
        'WNK1', 'WNK4', 'KLHL3', 'CUL3'],
    'Cardiovascular - Stroke / Cerebrovascular': [
        'NOTCH3', 'HTRA1', 'COL4A1', 'COL4A2', 'ACTA2', 'MYH11', 'TGFBR1', 'TGFBR2',
        'F2', 'F5', 'PROS1', 'PROC', 'SERPINC1'],
    'Type 2 Diabetes / MODY': [
        'TCF7L2', 'PPARG', 'KCNJ11', 'CDKAL1', 'FTO', 'SLC30A8', 'HHEX', 'IDE',
        'CDKN2A', 'CDKN2B', 'GCK', 'HNF1A', 'HNF4A', 'HNF1B', 'INS', 'ABCC8', 'PDX1',
        'KLF11', 'PAX4', 'BLK', 'CEL', 'NEUROD1'],
    'Obesity / Metabolic Syndrome': [
        'MC4R', 'LEP', 'LEPR', 'PCSK1', 'BDNF', 'SH2B1', 'FTO', 'TMEM18', 'POMC', 'NTRK2'],
    'Liver - NAFLD / Hemochromatosis / Wilson': [
        'PNPLA3', 'TM6SF2', 'MBOAT7', 'GCKR', 'HSD17B13', 'HFE', 'HAMP', 'HJV',
        'TFR2', 'SLC40A1', 'ATP7B'],
    'Kidney - CKD / Hereditary Nephropathy': [
        'PKD1', 'PKD2', 'UMOD', 'MUC1', 'COL4A3', 'COL4A4', 'COL4A5', 'NPHS1', 'NPHS2',
        'WT1', 'CFH', 'CFI', 'CD46', 'APOL1'],
    'Hereditary Cancer - Breast / Ovarian': [
        'BRCA1', 'BRCA2', 'PALB2', 'RAD51C', 'RAD51D', 'ATM', 'BARD1', 'BRIP1', 'CDH1'],
    'Hereditary Cancer - Colorectal / Lynch': [
        'MLH1', 'MSH2', 'MSH6', 'EPCAM', 'APC', 'MUTYH', 'STK11', 'SMAD4', 'BMPR1A'],
    'Hereditary Cancer - Gastric / Upper GI': [
        'CDH1', 'CTNNA1', 'MAP3K6', 'STK11', 'FBXW7', 'RHOA', 'ARID1A'],
    'Hereditary Cancer - Thyroid / MEN': [
        'RET', 'MEN1', 'SDHB', 'SDHC', 'SDHD', 'NF1', 'PTEN', 'TP53', 'FOXE1'],
    'Lipid Metabolism / Atherosclerosis': [
        'APOE', 'APOA1', 'APOA5', 'APOC3', 'CETP', 'LIPA', 'LIPC', 'LPL',
        'ABCA1', 'ABCG8', 'ABCG5'],
    'Thrombophilia / Coagulation': [
        'F2', 'F5', 'F8', 'F9', 'VWF', 'PROS1', 'PROC', 'SERPINC1', 'SERPINE1'],
    'Connective Tissue / Aortic Disease': [
        'FBN1', 'FBN2', 'TGFBR1', 'TGFBR2', 'SMAD3', 'COL3A1', 'COL5A1', 'COL5A2',
        'ACTA2', 'MYH11'],
    'Neurological / Psychiatric - Monogenic': [
        'APP', 'PSEN1', 'PSEN2', 'APOE', 'TREM2', 'SORL1', 'ABCA7', 'CLU', 'BIN1', 'CR1',
        'GRN', 'MAPT', 'LRRK2', 'PARK2', 'PINK1', 'SNCA'],
    'Autoimmune - Lupus / Rheumatoid Arthritis': [
        'DNASE1L3', 'TREX1', 'C1S', 'C1R', 'PTPN22', 'CTLA4', 'STAT4', 'IRF5', 'TNFAIP3'],
}

MIXED_PANELS = {
    'Cardiovascular - Hypertension', 'Type 2 Diabetes / MODY',
    'Obesity / Metabolic Syndrome', 'Liver - NAFLD / Hemochromatosis / Wilson',
    'Lipid Metabolism / Atherosclerosis', 'Neurological / Psychiatric - Monogenic',
    'Autoimmune - Lupus / Rheumatoid Arthritis',
}

N_PANELS = len(PANELS)
ALL_PANEL_GENES = {g for genes in PANELS.values() for g in genes}
FUNC_CLASSES = ['nonsynonymous SNV', 'stopgain', 'stoploss',
                'frameshift deletion', 'frameshift insertion']
print(f'{N_PANELS} gene panels, {len(ALL_PANEL_GENES)} unique genes')


def carriers_in(index_subset):
    if len(index_subset) == 0:
        return set()
    sub = (dos_plp.loc[index_subset] > 0) & (plo_plp.loc[index_subset] > 0)
    return set(sub.columns[sub.any(axis=0)])


def panel_tables(frame):
    """Per-panel variant counts and carrier frequencies for one P/LP variant set."""
    panel_rows, carrier_rows = [], []
    for panel, genes in PANELS.items():
        gset = set(genes)
        sub = frame[gene_mask(frame, gset)] if len(frame) else frame
        hit = sorted({g for gf in sub['Gene.refGene'] for g in genes_of(gf) & gset})
        func = sub['ExonicFunc.refGene'].astype('string').fillna('.').value_counts().to_dict()
        named = {k: int(func.get(k, 0)) for k in FUNC_CLASSES}
        other = len(sub) - sum(named.values())
        k = len(carriers_in(sub.index))
        lo, hi = wilson_ci(k, N_COHORT)
        composition = ('monogenic genes with common-variant risk loci'
                       if panel in MIXED_PANELS else 'monogenic genes')
        row = {'Disease panel': panel, 'Panel composition': composition,
               'Panel genes': len(genes), 'Genes with P/LP variants': len(hit),
               'Genes hit': ', '.join(hit), 'Total P/LP variants': len(sub)}
        row.update(named)
        row['Other / splicing / unannotated'] = other
        row['Carriers (n)'] = k
        panel_rows.append(row)
        carrier_rows.append({
            'Disease panel': panel, 'Panel composition': composition,
            'Panel genes': len(genes), 'P/LP variant sites': len(sub), 'Carriers (n)': k,
            'Carrier freq (%)': round(k / N_COHORT * 100, 2),
            '95% CI lower (%)': round(lo * 100, 2), '95% CI upper (%)': round(hi * 100, 2),
            '1 in N': one_in_n(k, N_COHORT, hi)})
    summary = (pd.DataFrame(panel_rows)
               .sort_values('Total P/LP variants', ascending=False).reset_index(drop=True))
    carriers = (pd.DataFrame(carrier_rows)
                .sort_values('Carrier freq (%)', ascending=False).reset_index(drop=True))
    assert (summary[FUNC_CLASSES + ['Other / splicing / unannotated']].sum(axis=1)
            == summary['Total P/LP variants']).all()
    return summary, carriers


panel_summary, df_carriers = panel_tables(df_rep)
panel_summary_all, df_carriers_all = panel_tables(df_plp)

df_panel = df_rep[gene_mask(df_rep, ALL_PANEL_GENES)].copy() if len(df_rep) else df_rep.copy()
if len(df_panel):
    df_panel['Panel'] = df_panel['Gene.refGene'].apply(
        lambda gf: '; '.join(sorted({p.split(' - ')[-1] for p, gs in PANELS.items()
                                     if genes_of(gf) & set(gs)})))
n_plp_panel = len(df_panel)
panel_carriers = carriers_in(df_panel.index)

df_curated_check = pd.DataFrame([
    {'Key': k, 'Transcript': e[0], 'HGVS_c': e[1], 'HGVS_p': e[2],
     'Expected REF>ALT': f'{e[3]}>{e[4]}'}
    for k, e in CURATED_HGVS.items()])

# detection power of a 219-individual cohort
df_power = pd.DataFrame([{
    'True allele frequency': af,
    'Expected alternate alleles': round(N_CHR_KZ * af, 2),
    'P(detect >=1 carrier)': round(1 - (1 - af) ** N_CHR_KZ, 3),
    'P(detect >=2 carriers)': round(
        1 - (1 - af) ** N_CHR_KZ - N_CHR_KZ * af * (1 - af) ** (N_CHR_KZ - 1), 3),
} for af in [0.0005, 0.001, 0.002, 0.005, 0.01, 0.02, 0.05]])
af_80 = 1 - 0.2 ** (1 / N_CHR_KZ)

print(f'P/LP variants in the panel genes: {n_plp_panel}')
print(f'individuals with at least one panel variant: {len(panel_carriers)} of {N_COHORT} '
      f'({len(panel_carriers)/N_COHORT*100:.1f}%)')
print(f'minimum allele frequency detected with 80% power: {af_80*100:.3f}%')
display(df_carriers[df_carriers['P/LP variant sites'] > 0])

## 5. ACMG Secondary Findings v3.3

P/LP variants in the 84 genes of ACMG SF v3.3 (Miller et al. *Genet Med* 2023 and the v3.3
update) are tabulated with allele counts. Where v3.3 attaches a reporting footnote to a gene,
the footnote text is carried in `ACMG_v33_footnote`; those genes are reportable only under the
stated zygosity or phenotype condition, which this descriptive analysis does not evaluate.

In [ ]:
df_acmg = df_rep[gene_mask(df_rep, ACMG_SF_V33)].copy() if len(df_rep) else df_rep.copy()

if df_acmg.empty:
    n_acmg_genes = n_acmg_footnoted = 0
    acmg_carriers = set()
    print('no P/LP variants in ACMG SF v3.3 genes at the reported review status')
else:
    df_acmg['ACMG_gene'] = df_acmg['Gene.refGene'].apply(
        lambda gf: ', '.join(sorted(genes_of(gf) & ACMG_SF_V33)))
    df_acmg['New_in_v3.3'] = df_acmg['Gene.refGene'].apply(
        lambda gf: bool(genes_of(gf) & ACMG_V33_NEW))
    df_acmg['ACMG_v33_footnote'] = df_acmg['Gene.refGene'].apply(
        lambda gf: '; '.join(ACMG_FOOTNOTE[g] for g in sorted(genes_of(gf))
                             if g in ACMG_FOOTNOTE) or pd.NA)
    n_acmg_genes = df_acmg['ACMG_gene'].str.split(',').explode().str.strip().nunique()
    n_acmg_footnoted = int(df_acmg['ACMG_v33_footnote'].notna().sum())
    acmg_carriers = carriers_in(df_acmg.index)

    print(f'ACMG SF v3.3 genes queried : {len(ACMG_SF_V33)}')
    print(f'genes with P/LP variants   : {n_acmg_genes}')
    print(f'P/LP variants in SF genes  : {len(df_acmg)}')
    print(f'  in genes carrying a v3.3 reporting footnote: {n_acmg_footnoted}')
    print(f'individuals with at least one SF variant: {len(acmg_carriers)} of {N_COHORT} '
          f'({len(acmg_carriers)/N_COHORT*100:.1f}%)')
    display(df_acmg[['Gene.refGene', 'avsnp151', 'HGVS_c', 'HGVS_p', 'HGVS_source',
                     'CLNSIG', 'ClinVar_stars', 'n_het', 'n_hom_alt', 'n_carriers',
                     'ACMG_v33_footnote']].reset_index(drop=True))

## 6. Trait loci, APOE and lactase persistence

Effect alleles are declared in `EFFECT_ALLELE_IS_REF` and polarisation is applied to the cohort
and to the reference panels together.

APOE alleles are inferred from the two-locus rs429358 / rs7412 genotype. Genotypes are unphased,
so a double heterozygote is recorded as e2/e4, the far more common of the two configurations
consistent with that genotype. Lactase persistence markers are grouped into haplotype blocks and
dosage within a block is the maximum of its markers, not their sum.

In [ ]:
import scipy
from scipy import stats as _st

TRAIT_SNPS = {
    'rs12913832': ('HERC2/OCA2', 'intron 86 enhancer', 'Pigmentation - Eye',
                   'regulatory switch for OCA2 transcription'),
    'rs16891982': ('SLC45A2', 'p.Phe374Leu', 'Pigmentation - Skin/Eye',
                   'membrane-associated transporter, melanin content'),
    'rs1393350': ('TYR', 'p.Arg402Gln', 'Pigmentation - Eye',
                  'temperature-sensitive tyrosinase variant'),
    'rs1042602': ('TYR', 'p.Ser192Tyr', 'Pigmentation - Hair/Skin',
                  'reduced tyrosinase activity'),
    'rs683': ('TYRP1', 'p.Ala530Val', 'Pigmentation - Hair',
              'tyrosinase-related protein 1, eumelanin synthesis'),
    'rs1426654': ('SLC24A5', 'p.Ala111Thr', 'Pigmentation - Skin',
                  'cation exchanger, skin melanin'),
    'rs1042725': ('HMGA2', "3' UTR", 'Anthropometric - Height',
                  'chromatin architectural factor, adult stature'),
    'rs9939609': ('FTO', 'intron 1', 'Anthropometric - BMI',
                  'first replicated common BMI locus'),
    'rs1421085': ('FTO', 'intron 1', 'Anthropometric - BMI',
                  'adipocyte browning switch'),
    'rs4988235': ('MCM6/LCT', 'LCT -13910C>T', 'Dietary - Lactase persistence',
                  'enhancer variant maintaining LCT transcription into adulthood'),
    'rs182549': ('MCM6/LCT', 'LCT -22018G>A', 'Dietary - Lactase persistence',
                 'second marker on the same West Eurasian persistence haplotype'),
    'rs41380347': ('MCM6/LCT', 'LCT -13915T>G', 'Dietary - Lactase persistence',
                   'Arabian persistence allele, independent origin'),
    'rs145946881': ('MCM6/LCT', 'LCT -14010G>C', 'Dietary - Lactase persistence',
                    'East African persistence allele, independent origin'),
    'rs41525747': ('MCM6/LCT', 'LCT -13907C>G', 'Dietary - Lactase persistence',
                   'North East African persistence allele, independent origin'),
    'rs1229984': ('ADH1B', 'p.His48Arg', 'Dietary - Alcohol metabolism',
                  'ethanol oxidation rate'),
    'rs698': ('ADH1C', 'p.Ile350Val', 'Dietary - Alcohol metabolism',
              'ethanol oxidation rate'),
    'rs671': ('ALDH2', 'p.Glu504Lys', 'Dietary - Alcohol metabolism',
              'aldehyde dehydrogenase activity'),
    'rs17822931': ('ABCC11', 'p.Gly180Arg', 'Secretion - Earwax type',
                   'wet versus dry cerumen'),
    'rs8176719': ('ABO', 'c.261delG', 'Blood group - ABO',
                  'the reference deletion allele defines blood group O'),
    'rs1800562': ('HFE', 'p.Cys282Tyr', 'Iron metabolism',
                  'principal hereditary haemochromatosis allele'),
    'rs1799945': ('HFE', 'p.His63Asp', 'Iron metabolism', 'milder HFE allele'),
    'rs2228570': ('VDR', 'FokI', 'Vitamin D - Receptor',
                  'alternative start codon usage'),
    'rs1800497': ('ANKK1', 'p.Glu713Lys', 'Neurobehavioral - Dopamine',
                  'striatal dopamine D2 receptor availability'),
    'rs429358': ('APOE', 'epsilon determinant 1', 'Disease risk - Alzheimer/Lipids',
                 'one of the two codons defining the APOE epsilon alleles'),
    'rs7412': ('APOE', 'epsilon determinant 2', 'Disease risk - Alzheimer/Lipids',
               'the second APOE epsilon codon'),
    'rs7903146': ('TCF7L2', 'intron 3', 'Disease risk - Type 2 Diabetes',
                  'largest common-variant effect on type 2 diabetes risk'),
}

LACTASE_LP_VARIANTS = ['rs4988235', 'rs182549', 'rs41380347', 'rs145946881', 'rs41525747']
LACTASE_BLOCKS = [['rs4988235', 'rs182549'], ['rs41380347'], ['rs145946881'],
                  ['rs41525747']]


def query_rsids(catalogue):
    """Frequencies, carrier counts and intervals for one curated rsID catalogue."""
    out = df_raw[df_raw['avsnp151'].astype('string').isin(list(catalogue))].copy()
    if out.empty:
        return out, None, None
    m, _ = in_paralogue_gene(out)
    out = out[~m].copy()
    out = out.drop_duplicates(subset=['Chr', 'Start', 'Ref', 'Alt']).copy()
    out['Curated_gene'] = out['avsnp151'].map(lambda r: catalogue[r][0])
    out['Variant'] = out['avsnp151'].map(lambda r: catalogue[r][1])
    out['Category'] = out['avsnp151'].map(lambda r: catalogue[r][2])
    out['Note'] = out['avsnp151'].map(lambda r: catalogue[r][3])
    out, dos, plo, _ = add_counts(out, gt_cols, quiet=True)

    out['Site_FILTER'] = (out[filter_col].astype('string') if filter_col
                          else pd.Series('not recorded', index=out.index, dtype='string'))
    out['Site_PASS'] = (out['Site_FILTER'].str.upper() == 'PASS' if filter_col else True)

    is_ref = out['avsnp151'].astype('string').isin(EFFECT_ALLELE_IS_REF)
    af_alt = pd.to_numeric(out[AF_COL], errors='coerce')
    out['Effect_allele'] = np.where(is_ref, out['Ref'], out['Alt'])
    out['Effect_allele_is_REF'] = is_ref
    out['KAZ_AF'] = np.where(is_ref, 1 - af_alt, af_alt).round(4)
    out['KAZ_AF_complement'] = (1 - out['KAZ_AF']).round(4)
    out['Reference_note'] = out['avsnp151'].astype('string').map(REFERENCE_NOTE)
    out['Genomic_change'] = out['Ref'].astype(str) + '>' + out['Alt'].astype(str)

    h = out.apply(lambda r: pd.Series(
        curated_hgvs_for(r.get('avsnp151'), r.get('Ref'), r.get('Alt'),
                         r.get('Chr'), r.get('Start')),
        index=['Transcript', 'HGVS_c', 'HGVS_p', 'HGVS_verdict']), axis=1)
    for c in ['Transcript', 'HGVS_c', 'HGVS_p']:
        out[c] = h[c]

    eff = effect_dosage(out, dos, plo)
    out['Effect_carriers (n)'] = (eff.fillna(0) > 0).sum(axis=1).astype(int)
    out['Effect_homozygotes (n)'] = (eff.fillna(0) == 2).sum(axis=1).astype(int)
    ci = out.apply(lambda r: wilson_ci(r['KAZ_AF'] * 2 * r['n_called'],
                                       2 * r['n_called']), axis=1, result_type='expand')
    out['KAZ_AF_CI_lower'] = ci[0].round(4)
    out['KAZ_AF_CI_upper'] = ci[1].round(4)
    return out, dos, plo


df_trait, dos_trait, plo_trait = query_rsids(TRAIT_SNPS)
found = set(df_trait['avsnp151'].astype(str)) if not df_trait.empty else set()
n_trait_nonpass = int((~df_trait['Site_PASS']).sum()) if not df_trait.empty else 0
print(f'trait loci queried {len(TRAIT_SNPS)}, observed {len(found)}, '
      f'site FILTER not PASS {n_trait_nonpass}')

# APOE
apoe = {}
for rs in ('rs429358', 'rs7412'):
    idx = df_trait.index[df_trait['avsnp151'].astype(str) == rs]
    if len(idx):
        apoe[rs] = dos_trait.loc[idx[0]].where(plo_trait.loc[idx[0]] > 0)

if len(apoe) == 2:
    d1, d2 = apoe['rs429358'], apoe['rs7412']
    call = {}
    for s in gt_cols:
        a, b = d1.get(s), d2.get(s)
        if pd.isna(a) or pd.isna(b):
            call[s] = 'no call'
            continue
        call[s] = {(0, 0): 'e3/e3', (0, 1): 'e2/e3', (0, 2): 'e2/e2',
                   (1, 0): 'e3/e4', (2, 0): 'e4/e4', (1, 1): 'e2/e4'}.get(
                       (int(a), int(b)), 'ambiguous')
    apoe_geno = pd.Series(call)
    resolved = apoe_geno[~apoe_geno.isin(['no call', 'ambiguous'])]
    counts_apoe = resolved.value_counts()
    called = int(len(resolved))
    e2 = counts_apoe.get('e2/e2', 0) * 2 + counts_apoe.get('e2/e3', 0) + counts_apoe.get('e2/e4', 0)
    e4 = counts_apoe.get('e4/e4', 0) * 2 + counts_apoe.get('e3/e4', 0) + counts_apoe.get('e2/e4', 0)
    e3 = 2 * called - e2 - e4
    df_apoe_geno = counts_apoe.rename_axis('APOE diplotype').reset_index(name='Individuals')
    df_apoe_geno['Note'] = np.where(
        df_apoe_geno['APOE diplotype'] == 'e2/e4',
        'unphased double heterozygote; e1/e3 not excluded', '')
    f2, f3, f4 = (round(e2 / (2 * called), 4), round(e3 / (2 * called), 4),
                  round(e4 / (2 * called), 4))
    ci_apoe = [wilson_ci(x, 2 * called) for x in (e2, e3, e4)]
    df_apoe = pd.DataFrame({
        'Allele': ['epsilon2', 'epsilon3', 'epsilon4'],
        'Alleles (n)': [e2, e3, e4],
        'KAZ_AF': [f2, f3, f4],
        'KAZ_AF_CI_lower': [round(lo, 4) for lo, _ in ci_apoe],
        'KAZ_AF_CI_upper': [round(hi, 4) for _, hi in ci_apoe]})
    print(f'APOE: {called} of {N_COHORT} individuals with a resolved diplotype, '
          f'{int((apoe_geno == "no call").sum())} with no call at one or both sites')
    display(df_apoe_geno)
    display(df_apoe)
else:
    df_apoe = df_apoe_geno = pd.DataFrame()

# lactase persistence
lact_rows, lp_dos = [], {}
for rs in LACTASE_LP_VARIANTS:
    idx = df_trait.index[df_trait['avsnp151'].astype(str) == rs]
    name, note = TRAIT_SNPS[rs][1], TRAIT_SNPS[rs][3]
    polarise = rs in EFFECT_ALLELE_IS_REF
    if len(idx) == 0:
        _, hi0 = wilson_ci(0, N_CHR_KZ)
        lact_rows.append({'rsID': rs, 'Variant': name, 'Gene': 'MCM6/LCT',
                          'Observed': 'not observed', 'Persistence allele': pd.NA,
                          'KAZ_AF': 0.0, '95% CI lower': 0.0,
                          '95% CI upper': round(hi0, 4), 'Carriers (n)': 0,
                          'Homozygotes (n)': 0, 'Called individuals': 0,
                          '1KG EUR': 0.0, '1KG EAS': 0.0,
                          'EUR_absent': True, 'EAS_absent': True, 'Note': note})
        continue
    r = df_trait.loc[idx[0]]
    lp_dos[rs] = dos_trait.loc[idx[0]].where(plo_trait.loc[idx[0]] > 0)

    def _pol(value):
        """Polarise to the persistence allele. Absent from the per-population panel
        is a zero count, not a missing measurement: those files list only sites
        polymorphic in the panel."""
        v = pd.to_numeric(pd.Series([value]), errors='coerce').iloc[0]
        if pd.isna(v):
            return 0.0, True
        return round(1 - v, 4) if polarise else round(v, 4), False

    eur, eur_absent = _pol(r.get('1000G_EUR'))
    eas, eas_absent = _pol(r.get('1000G_EAS'))
    lact_rows.append({
        'rsID': rs, 'Variant': name, 'Gene': 'MCM6/LCT', 'Observed': 'yes',
        'Persistence allele': r['Effect_allele'], 'KAZ_AF': r['KAZ_AF'],
        '95% CI lower': r['KAZ_AF_CI_lower'], '95% CI upper': r['KAZ_AF_CI_upper'],
        'Carriers (n)': int(r['Effect_carriers (n)']),
        'Homozygotes (n)': int(r['Effect_homozygotes (n)']),
        'Called individuals': int(r['n_called']),
        '1KG EUR': eur, '1KG EAS': eas,
        'EUR_absent': eur_absent, 'EAS_absent': eas_absent, 'Note': note})
df_lactase = pd.DataFrame(lact_rows)

if lp_dos:
    m = pd.DataFrame(lp_dos).fillna(0)
    block = pd.DataFrame({f'b{i}': m[[x for x in b if x in m.columns]].max(axis=1)
                          for i, b in enumerate(LACTASE_BLOCKS)
                          if any(x in m.columns for x in b)})
    total = block.sum(axis=1).clip(upper=2)
    n_lp, n_lp_hom = int((total > 0).sum()), int((total >= 2).sum())
else:
    n_lp = n_lp_hom = 0
lp_lo, lp_hi = wilson_ci(n_lp, N_COHORT)

print(f'lactase persistence: {int((df_lactase["Observed"] == "yes").sum())} of '
      f'{len(LACTASE_LP_VARIANTS)} alleles observed')
print(f'  at least one persistence allele: {n_lp} of {N_COHORT} '
      f'({n_lp/N_COHORT*100:.1f}%, 95% CI {lp_lo*100:.1f}-{lp_hi*100:.1f}%)')
print(f'  two persistence alleles: {n_lp_hom}')
display(df_lactase)

# LCT marker
LP_PRIMARY, LP_SECOND = 'rs4988235', 'rs182549'
PUBLISHED_LP = [
    ('Heyer et al. 2011, Hum Biol 83(3)', 'Kazakh', 0.30,
     'phenotypic lactase persistence 25-32%'),
    ('Regional review, Central Asia', 'Central Asian pooled', 0.13,
     'derived from a reported rs4988235-C frequency of about 87%'),
]

df_lp_concord = pd.DataFrame()
if {LP_PRIMARY, LP_SECOND} <= set(lp_dos):
    a, b = lp_dos[LP_PRIMARY], lp_dos[LP_SECOND]
    both = a.notna() & b.notna()
    a_i, b_i = a[both].astype(int), b[both].astype(int)
    n_both = int(((a_i > 0) & (b_i > 0)).sum())
    n_none = int(((a_i == 0) & (b_i == 0)).sum())
    only_1 = int(((a_i > 0) & (b_i == 0)).sum())
    only_2 = int(((a_i == 0) & (b_i > 0)).sum())
    n_dose = int(((a_i > 0) & (b_i > 0) & (a_i != b_i)).sum())
    df_lp_concord = pd.DataFrame([
        {'Genotype class': 'neither marker', 'Individuals': n_none},
        {'Genotype class': 'both markers', 'Individuals': n_both},
        {'Genotype class': f'{LP_SECOND} only', 'Individuals': only_2},
        {'Genotype class': f'{LP_PRIMARY} only', 'Individuals': only_1},
        {'Genotype class': 'both, dosage differs', 'Individuals': n_dose}])
    print(f'\nLCT marker concordance, {int(both.sum())} individuals called at both')
    display(df_lp_concord)
    # one-directional discordance is haplotype structure, not call error: every
    # -13910*T chromosome carries -22018*A, but not the reverse
    if only_1 + only_2:
        p_dir = _st.binomtest(only_1, only_1 + only_2, 0.5).pvalue
        print(f'  discordance is one-directional, {only_2} versus {only_1}; '
              f'binomial test against symmetric call error p = {p_dir:.4f}')
        print(f'  {LP_PRIMARY} is nested within {LP_SECOND}, so the primary marker '
              'gives the conservative estimate')
else:
    n_disc, r = 0, np.nan

if LP_PRIMARY in lp_dos:
    prim = lp_dos[LP_PRIMARY]
    n_called_p = int(prim.notna().sum())
    ac_p = int(prim.fillna(0).sum())
    n_lp_primary = int((prim.fillna(0) > 0).sum())
    n_lp_hom_primary = int((prim.fillna(0) >= 2).sum())
else:
    n_called_p = ac_p = n_lp_primary = n_lp_hom_primary = 0
af_primary = ac_p / max(2 * n_called_p, 1)
af_lo_p, af_hi_p = wilson_ci(ac_p, 2 * n_called_p)
lp_lo_p, lp_hi_p = wilson_ci(n_lp_primary, N_COHORT)

df_lp_published = pd.DataFrame(
    [{'Source': s, 'Group': g, '-13910*T frequency': f, 'Note': n}
     for s, g, f, n in PUBLISHED_LP]
    + [{'Source': f'{COHORT}, this study', 'Group': 'Kazakh',
        '-13910*T frequency': round(af_primary, 4),
        'Note': f'95% CI {af_lo_p:.4f}-{af_hi_p:.4f}, n = {N_COHORT}'}])

print(f'\nprimary marker {LP_PRIMARY}: allele frequency {af_primary:.4f} '
      f'(95% CI {af_lo_p:.4f}-{af_hi_p:.4f})')
print(f'  carriers {n_lp_primary} of {N_COHORT} ({n_lp_primary / N_COHORT * 100:.1f}%, '
      f'95% CI {lp_lo_p * 100:.1f}-{lp_hi_p * 100:.1f}%), homozygotes {n_lp_hom_primary}')
print(f'  block rule (either marker) gives {n_lp} carriers, reported as a sensitivity value')
display(df_lp_published)

## 7. Pharmacogenomic loci

Allele frequencies at prescribing-relevant positions. Star alleles are haplotypes and these
genotypes are unphased, so no diplotype and no metaboliser phenotype is derived; *NAT2* is
reported as single-SNP frequencies at the seven signature positions. Genes with a high-identity
paralogue and HLA tag SNPs are out of scope.

A queried allele that is absent from the callset is reported with the Wilson upper bound the
sample size supports. That bound assumes the position was callable in every individual, which
was not verified site by site.

In [ ]:
# CPIC gene-drug pair levels, from the two lists at https://www.clinpgx.org/cpic/pairs
# (cpic-genes-drugs.tsv and cpic-provisional-genes-drugs.tsv, downloaded 08 Sep 2026).
#
# CPIC assigns a level to a gene-DRUG PAIR, not to a variant, and a level is
# provisional until the pair has been reviewed in a CPIC guideline. Recording the
# drug and the status is what makes the entry checkable against the source; a bare
# letter cannot be verified by anyone, including us.
#   (gene, variant, therapeutic area, allele function, CPIC level, status, CPIC drug)
PGX_SNPS_FULL = {
    'rs9923231': ('VKORC1', '-1639G>A', 'Anticoagulation', 'reduced VKORC1 expression',
                  'A', 'final', 'warfarin'),
    'rs1799853': ('CYP2C9', '*2 p.Arg144Cys', 'Anticoagulation', 'decreased function',
                  'A', 'final', 'warfarin'),
    'rs1057910': ('CYP2C9', '*3 p.Ile359Leu', 'Anticoagulation', 'no function',
                  'A', 'final', 'warfarin'),
    'rs2108622': ('CYP4F2', '*3 p.Val433Met', 'Anticoagulation',
                  'reduced vitamin K1 oxidation', 'A', 'final', 'warfarin'),
    # F5 and F2 are thrombophilia risk factors, an indication for anticoagulation and
    # not modifiers of the drug response. F5 does appear in CPIC, but paired with
    # eltrombopag and hormonal contraceptives; F2 is absent from both lists.
    'rs6025': ('F5', 'c.1601G>A (Leiden)', 'Thrombophilia',
               'activated protein C resistance', 'none', 'not in CPIC for this drug',
               'no anticoagulant pair'),
    'rs1799963': ('F2', 'c.*97G>A (G20210A)', 'Thrombophilia',
                  'increased prothrombin level', 'none', 'not in CPIC', 'none'),
    'rs4244285': ('CYP2C19', '*2 c.681G>A', 'Antiplatelet', 'no function',
                  'A', 'final', 'clopidogrel'),
    'rs4986893': ('CYP2C19', '*3 p.Trp212Ter', 'Antiplatelet', 'no function',
                  'A', 'final', 'clopidogrel'),
    'rs12248560': ('CYP2C19', '*17 c.-806C>T', 'Antiplatelet', 'increased function',
                   'A', 'final', 'clopidogrel'),
    'rs4149056': ('SLCO1B1', '*5 p.Val174Ala', 'Statins',
                  'decreased transporter function', 'A', 'final', 'simvastatin'),
    'rs2306283': ('SLCO1B1', '*1b p.Asn130Asp', 'Statins', 'haplotype partner of *5',
                  'A', 'final', 'simvastatin'),
    'rs2231142': ('ABCG2', 'p.Gln141Lys', 'Statins', 'decreased function',
                  'A', 'final', 'rosuvastatin'),          # was B
    'rs1799971': ('OPRM1', 'p.Asn40Asp', 'Opioids and antidepressants',
                  'altered receptor binding', 'C', 'final', 'codeine'),   # was B
    'rs776746': ('CYP3A5', '*3 c.219-237A>G', 'Immunosuppression',
                 'no function, non-expresser', 'A', 'final', 'tacrolimus'),
    'rs35599367': ('CYP3A4', '*22 c.522-191C>T', 'Immunosuppression',
                   'decreased function', 'C', 'provisional', 'tacrolimus'),
    'rs1800460': ('TPMT', '*3B p.Ala154Thr', 'Thiopurines', 'no function',
                  'A', 'final', 'azathioprine'),
    'rs1142345': ('TPMT', '*3C p.Tyr240Cys', 'Thiopurines', 'no function',
                  'A', 'final', 'azathioprine'),
    'rs116855232': ('NUDT15', '*3 p.Arg139Cys', 'Thiopurines', 'no function',
                    'A', 'final', 'azathioprine'),
    'rs3918290': ('DPYD', '*2A c.1905+1G>A', 'Fluoropyrimidines', 'no function',
                  'A', 'final', 'fluorouracil'),
    'rs55886062': ('DPYD', '*13 p.Ile560Ser', 'Fluoropyrimidines', 'no function',
                   'A', 'final', 'fluorouracil'),
    'rs67376798': ('DPYD', 'c.2846A>T p.Asp949Val', 'Fluoropyrimidines',
                   'decreased function', 'A', 'final', 'fluorouracil'),
    'rs75017182': ('DPYD', 'HapB3 c.1129-5923C>G', 'Fluoropyrimidines',
                   'decreased function', 'A', 'final', 'fluorouracil'),
    'rs56038477': ('DPYD', 'HapB3 c.1236G>A', 'Fluoropyrimidines',
                   'HapB3 haplotype marker', 'A', 'final', 'fluorouracil'),
    'rs1801159': ('DPYD', '*5 p.Ile543Val', 'Fluoropyrimidines',
                  'normal function under CPIC', 'A', 'final', 'fluorouracil'),
    # level A, but provisional: CPIC has no irinotecan guideline, so the pair has not
    # been reviewed. Testing is Recommended on the FDA label and covered by DPWG.
    'rs4148323': ('UGT1A1', '*6 p.Gly71Arg', 'Irinotecan', 'decreased function',
                  'A', 'provisional', 'irinotecan'),
    # CYP2C8 is in CPIC only with NSAIDs and rosiglitazone; there is no platinum or
    # taxane pair, so the level cannot be carried over to this therapeutic area.
    'rs10509681': ('CYP2C8', '*3 p.Lys399Arg', 'Platinum and taxanes',
                   'decreased function', 'none', 'not in CPIC for this drug',
                   'no platinum or taxane pair'),
    'rs11572080': ('CYP2C8', '*3 p.Arg139Lys', 'Platinum and taxanes',
                   '*3 haplotype marker', 'none', 'not in CPIC for this drug',
                   'no platinum or taxane pair'),
    'rs1801516': ('ATM', 'p.Asp1853Asn', 'Platinum and taxanes',
                  'common coding variant', 'none', 'not in CPIC', 'none'),
    'rs13181': ('ERCC2', 'p.Lys751Gln', 'Platinum and taxanes',
                'nucleotide excision repair', 'none', 'not in CPIC', 'none'),
    'rs1801133': ('MTHFR', 'c.665C>T p.Ala222Val', 'Folate and methotrexate',
                  'reduced enzyme thermostability', 'C', 'provisional', 'methotrexate'),
    'rs1801131': ('MTHFR', 'c.1286A>C p.Glu429Ala', 'Folate and methotrexate',
                  'reduced enzyme activity', 'C', 'provisional', 'methotrexate'),
    'rs1801279': ('NAT2', '191G>A (*14 signature)', 'Antituberculosis',
                  'slow acetylation cluster', 'C', 'provisional', 'isoniazid'),
    'rs1041983': ('NAT2', '282C>T (*13 signature)', 'Antituberculosis',
                  'rapid acetylation cluster', 'C', 'provisional', 'isoniazid'),
    'rs1801280': ('NAT2', '341T>C (*5 signature)', 'Antituberculosis',
                  'slow acetylation cluster', 'C', 'provisional', 'isoniazid'),
    'rs1799929': ('NAT2', '481C>T (*11 signature)', 'Antituberculosis',
                  'rapid acetylation cluster', 'C', 'provisional', 'isoniazid'),
    'rs1799930': ('NAT2', '590G>A (*6 signature)', 'Antituberculosis',
                  'slow acetylation cluster', 'C', 'provisional', 'isoniazid'),
    'rs1208': ('NAT2', '803A>G (*12 signature)', 'Antituberculosis',
               'rapid acetylation cluster', 'C', 'provisional', 'isoniazid'),
    'rs1799931': ('NAT2', '857G>A (*7 signature)', 'Antituberculosis',
                  'slow acetylation cluster', 'C', 'provisional', 'isoniazid'),
    # the CPIC peginterferon guideline has been retired; the frequency is still
    # informative, the recommendation is not
    'rs12979860': ('IFNL3', 'c.-1291C>T', 'Antivirals',
                   'interferon response in hepatitis C', 'Retired', 'final',
                   'peginterferon alfa-2a'),
}
CPIC_ACCESSED = '2026-09-08'

df_pgx, dos_pgx, plo_pgx = query_rsids({rs: v[:4] for rs, v in PGX_SNPS_FULL.items()})

if df_pgx.empty:
    df_pgx_absent = pd.DataFrame()
    found_pgx = set()
else:
    df_pgx['Function'] = df_pgx['avsnp151'].map(lambda r: PGX_SNPS_FULL[r][3])
    df_pgx['CPIC_level'] = df_pgx['avsnp151'].map(lambda r: PGX_SNPS_FULL[r][4])
    df_pgx['CPIC_status'] = df_pgx['avsnp151'].map(lambda r: PGX_SNPS_FULL[r][5])
    df_pgx['CPIC_drug'] = df_pgx['avsnp151'].map(lambda r: PGX_SNPS_FULL[r][6])
    # only a level A pair that CPIC has actually reviewed is treated as actionable
    df_pgx['CPIC_actionable'] = ((df_pgx['CPIC_level'] == 'A')
                                 & (df_pgx['CPIC_status'] == 'final'))
    found_pgx = set(df_pgx['avsnp151'].astype(str))
    missing_pgx = sorted(set(PGX_SNPS_FULL) - found_pgx)
    print(f'pharmacogenomic loci queried {len(PGX_SNPS_FULL)}, observed {len(found_pgx)}, '
          f'therapeutic areas {df_pgx["Category"].nunique()}, '
          f'site FILTER not PASS {int((~df_pgx["Site_PASS"]).sum())}')
    print(f'CPIC pair lists accessed {CPIC_ACCESSED}: '
          + ', '.join(f'{k} {v}' for k, v in
                      df_pgx.groupby(["CPIC_level", "CPIC_status"]).size().items()))
    print(f'  level A and reviewed in a guideline: {int(df_pgx["CPIC_actionable"].sum())} '
          'of the observed loci')
    if missing_pgx:
        _, hi0 = wilson_ci(0, N_CHR_KZ)
        df_pgx_absent = pd.DataFrame([
            {'rsID': rs, 'Gene': PGX_SNPS_FULL[rs][0], 'Variant': PGX_SNPS_FULL[rs][1],
             'Therapeutic area': PGX_SNPS_FULL[rs][2], 'CPIC level': PGX_SNPS_FULL[rs][4],
             'CPIC status': PGX_SNPS_FULL[rs][5], 'CPIC drug': PGX_SNPS_FULL[rs][6],
             'KAZ_AF': 0.0, '95% CI upper': round(hi0, 4)}
            for rs in missing_pgx])
        print(f'  {len(missing_pgx)} queried allele(s) not observed, each bounded above '
              f'{hi0*100:.2f}% assuming complete callability')
        display(df_pgx_absent)
    else:
        df_pgx_absent = pd.DataFrame()

    NAT2_CODING_POS = {'rs1801279': 191, 'rs1041983': 282, 'rs1801280': 341,
                       'rs1799929': 481, 'rs1799930': 590, 'rs1208': 803,
                       'rs1799931': 857}
    nat2 = df_pgx[df_pgx['Curated_gene'] == 'NAT2']
    obs = dict(zip(nat2['avsnp151'].astype(str), nat2['Start']))
    if len(obs) >= 2:
        anchor = min(obs, key=lambda k: NAT2_CODING_POS[k])
        off = obs[anchor] - NAT2_CODING_POS[anchor]
        ok = sum(1 for rs, pos in obs.items() if pos == NAT2_CODING_POS[rs] + off)
        print(f'NAT2 signature SNPs observed {len(obs)} of 7, '
              f'spacing consistent for {ok} of {len(obs)}')

## 8. Reference frequencies and variant identity

1000 Genomes Phase 3 and gnomAD v2.1.1 frequencies are tabulated alongside the cohort and
polarised to the same allele. gnomAD is drawn from genomes for non-coding sites and exomes for
coding sites, recorded per row.

Every curated locus is checked against `VARIANT_IDENTITY`: GRCh37 coordinate, REF and ALT, the
polarised effect allele, and the published effect-allele frequency as an orientation check.
Loci that fail are listed rather than silently corrected.

In [ ]:
PANEL_1KG = [('1KG_EUR', '1000G_EUR'), ('1KG_EAS', '1000G_EAS'), ('1KG_SAS', '1000G_SAS')]
GNOMAD_POPS = [('NFE', 'AF_nfe', 'AF_nfe.1'), ('EAS', 'AF_eas', 'AF_eas.1'),
               ('SAS', 'AF_sas', None)]

def validate_registry(reg):
    """The strand and coding fields are exported but nothing tests them; this does.

    A coding change must reconcile with the genomic alleles under the recorded
    strand. Two of the three fields cross-validate, so a failure usually means the
    strand label is wrong rather than the alleles.
    """
    comp = {'A': 'T', 'T': 'A', 'G': 'C', 'C': 'G', '-': '-'}
    sub = re.compile(r'([ACGT])>([ACGT])')
    bad = []
    for rs, e in reg.items():
        if e['effect'] not in (e['ref'], e['alt']):
            bad.append((rs, 'effect allele is neither ref nor alt'))
        m = sub.search(str(e['coding']))
        if not m or '-' in (e['ref'], e['alt']):
            continue
        a, b = m.groups()
        plus = {a, b} == {e['ref'], e['alt']}
        minus = {comp[a], comp[b]} == {e['ref'], e['alt']}
        if not (plus if e['strand'] == '+' else minus):
            bad.append((rs, f"coding {e['coding']} vs {e['ref']}>{e['alt']} fits "
                            + ('the + strand' if plus else 'the - strand' if minus
                               else 'neither strand')))
    for rs, why in bad:
        print(f'  registry: {rs}, {why}')
    return bad

def polarise_reference_frequencies(frame):
    if frame is None or frame.empty:
        return frame
    out = frame.copy()
    polar = out['avsnp151'].astype('string').isin(EFFECT_ALLELE_IS_REF).to_numpy()
    for name, col in PANEL_1KG:
        if col not in out.columns:
            continue
        af = pd.to_numeric(out[col], errors='coerce')
        out[f'AF_{name}'] = pd.Series(np.where(polar, 1 - af, af),
                                      index=out.index).round(4)
    coding = out['Func.refGene'].astype('string').str.contains('exonic', case=False,
                                                               na=False)
    for name, ex_col, gen_col in GNOMAD_POPS:
        ex = pd.to_numeric(out.get(ex_col, pd.Series(np.nan, index=out.index)),
                           errors='coerce')
        gen = (pd.to_numeric(out.get(gen_col), errors='coerce')
               if gen_col and gen_col in out.columns
               else pd.Series(np.nan, index=out.index))
        chosen = ex if gen_col is None else ex.where(coding, gen)
        src = (pd.Series(np.where(coding, 'gnomAD v2.1.1 exomes',
                                  'gnomAD v2.1.1 genomes'), index=out.index) if gen_col
               else pd.Series('gnomAD v2.1.1 exomes', index=out.index))
        chosen = pd.Series(np.where(polar, 1 - chosen, chosen), index=out.index)
        out[f'AF_gnomAD_{name}'] = chosen.round(4)
        out[f'gnomAD_source_{name}'] = src.mask(chosen.isna(), 'not covered')
    return out


df_trait = polarise_reference_frequencies(df_trait)
df_pgx = polarise_reference_frequencies(df_pgx)
for _f, _n in ((df_trait, 'df_trait'), (df_pgx, 'df_pgx')):
    _miss = [c for c in ('AF_1KG_EUR', 'AF_1KG_EAS') if c not in _f.columns]
    assert not _miss, f'{_n} is missing {_miss}; every sheet and figure must read the '\
                      'polarised column, never the raw 1000G_* one'

df_panel_cmp = df_panel.copy()
if not df_panel_cmp.empty:
    for name, col in PANEL_1KG[:2]:
        if col in df_panel_cmp.columns:
            df_panel_cmp[f'AF_{name}'] = pd.to_numeric(df_panel_cmp[col],
                                                       errors='coerce').round(5)

def _v(chrom=None, pos=None, ref=None, alt=None, strand=None, coding=None,
       effect=None, eur=None, eas=None, source=''):
    return dict(chrom=chrom, pos=pos, ref=ref, alt=alt, strand=strand, coding=coding,
                effect=effect, eur=eur, eas=eas, source=source)

VARIANT_IDENTITY = {
    'rs9923231':  _v('chr16', 31107689, 'C', 'T', '-', 'c.-1639G>A', 'T', 0.39, 0.90, 'CPIC'),
    'rs1799853':  _v('chr10', 96702047, 'C', 'T', '+', 'c.430C>T (*2)', 'T', 0.13, 0.00,
                     'PharmVar'),
    'rs1057910':  _v('chr10', 96741053, 'A', 'C', '+', 'c.1075A>C (*3)', 'C', 0.07, 0.03,
                     'PharmVar'),
    'rs2108622':  _v('chr19', 15990431, 'C', 'T', '-', 'c.1297G>A (*3)', 'T', 0.29, 0.21,
                     'PharmGKB'),
    'rs6025':     _v('chr1', 169519049, 'T', 'C', '-', 'c.1601G>A', 'T', 0.02, 0.00,
                     'ClinVar, reference carries the variant allele'),
    'rs1799963':  _v('chr11', 46761055, 'G', 'A', '+', 'c.*97G>A', 'A', 0.01, 0.00,
                     'ClinVar'),
    'rs4244285':  _v('chr10', 96541616, 'G', 'A', '+', 'c.681G>A (*2)', 'A', 0.15, 0.31,
                     'PharmVar'),
    'rs4986893':  _v('chr10', 96540410, 'G', 'A', '+', 'c.636G>A (*3)', 'A', 0.00, 0.06,
                     'PharmVar'),
    'rs12248560': _v('chr10', 96521657, 'C', 'T', '+', 'c.-806C>T (*17)', 'T', 0.22, 0.02,
                     'PharmVar'),
    'rs4149056':  _v('chr12', 21331549, 'T', 'C', '+', 'c.521T>C (*5)', 'C', 0.16, 0.12,
                     'CPIC'),
    'rs2306283':  _v('chr12', 21329738, 'A', 'G', '+', 'c.388A>G (*1b)', 'G', 0.40, 0.75,
                     'CPIC'),
    'rs2231142':  _v('chr4', 89052323, 'G', 'T', '-', 'c.421C>A', 'T', 0.09, 0.29, 'CPIC'),
    'rs1799971':  _v('chr6', 154360797, 'A', 'G', '+', 'c.118A>G', 'G', 0.16, 0.39,
                     'PharmGKB'),
    'rs776746':   _v('chr7', 99270539, 'C', 'T', '-', 'c.219-237A>G (*3)', 'C', 0.94, 0.71,
                     'PharmVar, reference carries the *3 allele'),
    'rs35599367': _v('chr7', 99366316, 'G', 'A', '-', 'c.522-191C>T (*22)', 'A', 0.05, 0.00,
                     'PharmVar'),
    'rs1800460':  _v('chr6', 18139228, 'C', 'T', '-', 'c.460G>A (*3B)', 'T', 0.03, 0.00,
                     'CPIC'),
    'rs1142345':  _v('chr6', 18130918, 'T', 'C', '-', 'c.719A>G (*3C)', 'C', 0.03, 0.02,
                     'CPIC'),
    'rs116855232': _v('chr13', 48619855, 'C', 'T', '+', 'c.415C>T (*3)', 'T', 0.002, 0.10,
                      'CPIC'),
    'rs3918290':  _v('chr1', 97915614, 'C', 'T', '-', 'c.1905+1G>A (*2A)', 'T', 0.005, 0.00,
                     'CPIC'),
    'rs67376798': _v('chr1', 97547947, 'T', 'A', '-', 'c.2846A>T', 'A', 0.008, 0.00, 'CPIC'),
    'rs1801159':  _v('chr1', 97981395, 'T', 'C', '-', 'c.1627A>G (*5)', 'C', 0.19, 0.27,
                     'CPIC, normal function'),
    'rs56038477': _v('chr1', 98039419, 'C', 'T', '-', 'c.1236G>A', 'T', 0.024, 0.00, 'CPIC'),
    'rs75017182': _v('chr1', 98045449, 'G', 'C', '-', 'c.1129-5923C>G', 'C', 0.024, 0.00,
                     'CPIC'),
    'rs4148323':  _v('chr2', 234669144, 'G', 'A', '+', 'c.211G>A (*6)', 'A', 0.01, 0.14,
                     'PharmVar'),
    'rs10509681': _v('chr10', 96798749, 'T', 'C', '-', 'c.1196A>G (*3)', 'C', 0.12, 0.00,
                     'PharmGKB'),
    'rs11572080': _v('chr10', 96827030, 'C', 'T', '-', 'c.416G>A (*3)', 'T', 0.12, 0.00,
                     'PharmGKB'),
    'rs1801516':  _v('chr11', 108175462, 'G', 'A', '+', 'c.5557G>A', 'A', 0.15, 0.02,
                     'dbSNP'),
    'rs13181':    _v('chr19', 45854919, 'T', 'G', '-', 'c.2251A>C', 'G', 0.36, 0.08,
                     'dbSNP'),
    'rs1801133':  _v('chr1', 11856378, 'G', 'A', '-', 'c.665C>T', 'A', 0.36, 0.30, 'dbSNP'),
    'rs1801131':  _v('chr1', 11854476, 'T', 'G', '-', 'c.1286A>C', 'G', 0.31, 0.22, 'dbSNP'),
    'rs1801279':  _v('chr8', None, 'G', 'A', '+', '191G>A (*14)', 'A', 0.02, 0.00,
                     'NAT2 nomenclature'),
    'rs1041983':  _v('chr8', 18257795, 'C', 'T', '+', '282C>T (*13)', 'T', 0.31, 0.44,
                     'NAT2 nomenclature'),
    'rs1801280':  _v('chr8', 18257854, 'T', 'C', '+', '341T>C (*5)', 'C', 0.45, 0.04,
                     'NAT2 nomenclature'),
    'rs1799929':  _v('chr8', None, 'C', 'T', '+', '481C>T (*11)', 'T', 0.44, 0.04,
                     'NAT2 nomenclature'),
    'rs1799930':  _v('chr8', 18258103, 'G', 'A', '+', '590G>A (*6)', 'A', 0.28, 0.26,
                     'NAT2 nomenclature'),
    'rs1208':     _v('chr8', None, 'G', 'A', '+', '803A>G (*12)', 'G', 0.44, 0.04,
                     'NAT2 nomenclature, reference carries 803G'),
    'rs1799931':  _v('chr8', 18258370, 'G', 'A', '+', '857G>A (*7)', 'A', 0.02, 0.18,
                     'NAT2 nomenclature'),
    'rs12979860': _v('chr19', 39738787, 'C', 'T', '+', 'c.-1291C>T', 'T', 0.31, 0.08,
                     'PharmGKB'),
    'rs4988235':  _v('chr2', 136608646, 'G', 'A', '-', 'LCT -13910C>T', 'A', 0.51, 0.00,
                     'literature'),
    'rs182549':   _v('chr2', 136616754, 'C', 'T', '-', 'LCT -22018G>A', 'T', 0.51, 0.00,
                     'literature'),
    'rs1229984':  _v('chr4', 100239319, 'T', 'C', '-', 'c.143A>G (ADH1B*2)', 'T', 0.03, 0.70,
                     'dbSNP, reference carries the *2 allele'),
    'rs698':      _v('chr4', 100260789, 'T', 'C', '-', 'c.1045G>A (ADH1C*2)', 'C', 0.40,
                     0.08, 'dbSNP'),
    'rs671':      _v('chr12', 112241766, 'G', 'A', '+', 'c.1510G>A (ALDH2*2)', 'A', 0.00,
                     0.17, 'dbSNP'),
    'rs16891982': _v('chr5', 33951693, 'C', 'G', '+', 'c.1122C>G', 'G', 0.94, 0.01, 'dbSNP'),
    'rs1426654':  _v('chr15', 48426484, 'A', 'G', '+', 'c.331G>A', 'A', 0.997, 0.01,
                     'dbSNP, reference carries the derived allele'),
    'rs12913832': _v('chr15', 28365618, 'A', 'G', '-', 'HERC2 intron 86', 'G', 0.64, 0.00,
                     'ClinVar'),
    'rs1042602':  _v('chr11', 88911696, 'C', 'A', '+', 'c.575C>A', 'A', 0.37, 0.00, 'dbSNP'),
    'rs1393350':  _v('chr11', 89011046, 'G', 'A', '+', 'TYR intron', 'A', 0.24, 0.00,
                     'dbSNP'),
    'rs683':      _v('chr9', 12709305, 'C', 'A', '+', 'c.1589C>A', 'A', 0.63, 0.01, 'dbSNP'),
    'rs17822931': _v('chr16', 48258198, 'C', 'T', '-', 'c.538G>A', 'T', 0.14, 0.78, 'dbSNP'),
    'rs8176719':  _v('chr9', 136132908, '-', 'C', '-', 'c.261delG', 'C', 0.39, 0.38,
                     'dbSNP'),
    'rs1800562':  _v('chr6', 26093141, 'G', 'A', '+', 'c.845G>A', 'A', 0.05, 0.00,
                     'ClinVar'),
    'rs1799945':  _v('chr6', 26091179, 'C', 'G', '+', 'c.187C>G', 'G', 0.15, 0.03,
                     'ClinVar'),
    'rs2228570':  _v('chr12', 48272895, 'A', 'G', '-', 'VDR FokI', 'G', 0.62, 0.58, 'dbSNP'),
    'rs1800497':  _v('chr11', 113270828, 'G', 'A', '+', 'c.2137G>A', 'A', 0.19, 0.41,
                     'dbSNP'),
    'rs429358':   _v('chr19', 45411941, 'T', 'C', '+', 'c.388T>C', 'C', 0.15, 0.09,
                     'ClinVar'),
    'rs7412':     _v('chr19', 45412079, 'C', 'T', '+', 'c.526C>T', 'T', 0.06, 0.10,
                     'ClinVar'),
    'rs7903146':  _v('chr10', 114758349, 'C', 'T', '+', 'TCF7L2 intron 3', 'T', 0.30, 0.03,
                     'dbSNP'),
    'rs1042725':  _v('chr12', 66358347, 'C', 'T', '+', "HMGA2 3' UTR", 'C', 0.46, 0.18,
                     'dbSNP, reference carries the effect allele'),
    'rs9939609':  _v('chr16', 53820527, 'T', 'A', '+', 'FTO intron 1', 'A', 0.41, 0.17,
                     'dbSNP'),
    'rs1421085':  _v('chr16', 53800954, 'T', 'C', '+', 'FTO intron 1', 'C', 0.43, 0.17,
                     'dbSNP'),
}

validate_registry(VARIANT_IDENTITY)

AF_TOL = 0.12

def verify_identity(frame, label=''):
    if frame is None or frame.empty:
        return frame, pd.DataFrame()
    out, rows = frame.copy(), []
    for i, r in out.iterrows():
        rs = str(r.get('avsnp151'))
        exp = VARIANT_IDENTITY.get(rs)
        obs = f"{r.get('Chr')}:{int(r.get('Start'))} {r.get('Ref')}>{r.get('Alt')}"
        if exp is None:
            rows.append({'rsID': rs, 'Gene': r.get('Curated_gene'), 'Observed': obs,
                         'Expected': 'not in registry', 'Position': 'not checked',
                         'Alleles': 'not checked', 'Effect allele': 'not checked',
                         'Frequency check': 'not checked', 'Status': 'REVIEW',
                         'Strand': pd.NA, 'Coding change': pd.NA, 'Source': pd.NA})
            continue

        if exp['pos'] is None:
            pos_res = 'not checked'
        elif str(r.get('Chr')) == str(exp['chrom']) and int(r.get('Start')) == exp['pos']:
            pos_res = 'match'
        else:
            pos_res = f"MISMATCH, expected {exp['chrom']}:{exp['pos']}"

        o_ref, o_alt = str(r.get('Ref')), str(r.get('Alt'))
        direct = {o_ref, o_alt} == {exp['ref'], exp['alt']}
        comp = ({COMPLEMENT.get(o_ref, o_ref), COMPLEMENT.get(o_alt, o_alt)}
                == {exp['ref'], exp['alt']})
        all_res = ('match' if direct else
                   'COMPLEMENTARY, registry entry is on the other strand' if comp else
                   f"MISMATCH, expected {exp['ref']}>{exp['alt']}")

        o_eff = str(r.get('Effect_allele'))
        eff_res = ('match' if o_eff == exp['effect'] else
                   'COMPLEMENTARY' if o_eff == COMPLEMENT.get(exp['effect']) else
                   f"MISMATCH, expected {exp['effect']}")

        checks, verdicts = [], []
        for pop, col in (('eur', 'AF_1KG_EUR'), ('eas', 'AF_1KG_EAS')):
            e, o = exp[pop], pd.to_numeric(r.get(col), errors='coerce')
            if e is None or pd.isna(o):
                continue
            d, dc = abs(o - e), abs((1 - o) - e)
            checks.append(f'{pop.upper()} {o:.3f} vs {e:.3f}')
            verdicts.append('ok' if d <= AF_TOL else 'inverted' if dc < d else 'off')
        freq_res = ('not checked' if not checks else
                    'POSSIBLE INVERSION, ' + '; '.join(checks) if 'inverted' in verdicts else
                    'outside tolerance, ' + '; '.join(checks) if 'off' in verdicts else
                    'consistent, ' + '; '.join(checks))

        hard = [pos_res, all_res, eff_res]
        status = ('FAIL' if any(x.startswith(('MISMATCH', 'COMPLEMENTARY')) for x in hard)
                  or freq_res.startswith('POSSIBLE INVERSION')
                  else 'REVIEW' if any('not checked' in x for x in hard)
                  or freq_res.startswith('outside') else 'PASS')
        rows.append({'rsID': rs, 'Gene': r.get('Curated_gene'), 'Observed': obs,
                     'Expected': (f"{exp['chrom']}:{exp['pos']} {exp['ref']}>{exp['alt']}"
                                  if exp['pos'] else f"{exp['ref']}>{exp['alt']}"),
                     'Position': pos_res, 'Alleles': all_res, 'Effect allele': eff_res,
                     'Frequency check': freq_res, 'Status': status,
                     'Strand': exp['strand'], 'Coding change': exp['coding'],
                     'Source': exp['source']})

    report = pd.DataFrame(rows)
    out['Identity_status'] = report['Status'].values
    out['Gene_strand'] = report['Strand'].values
    out['Coding_change'] = report['Coding change'].values
    if label:
        print(f'{label}: ' + ', '.join(f'{k} {v}' for k, v in
                                       sorted(report['Status'].value_counts().items())))
        bad = report[report['Status'] != 'PASS']
        if not bad.empty:
            display(bad.drop(columns=['Strand', 'Coding change', 'Source']))
    return out, report


df_trait, id_trait = verify_identity(df_trait, 'identity, trait loci')
df_pgx, id_pgx = verify_identity(df_pgx, 'identity, pharmacogenomic loci')

# population comparison, after verify_identity
from scipy import stats as _st

N_CHR_REF = {'EUR': 1006, 'EAS': 1008, 'SAS': 978}   # 1000G Phase 3: 503, 504, 489
TEST_POPS = ['EUR', 'EAS']


def bh_q(pvals):
    p = np.asarray(pvals, dtype=float)
    q = np.full(p.shape, np.nan)
    ok = np.isfinite(p)
    if not ok.any():
        return q
    s = p[ok]
    o = np.argsort(s)
    m = len(s)
    adj = np.minimum.accumulate((s[o] * m / np.arange(1, m + 1))[::-1])[::-1]
    out = np.empty(m)
    out[o] = np.clip(adj, 0, 1)
    q[ok] = out
    return q


def _or_ci(k1, n1, k2, n2, z=1.96):
    """Odds ratio with a Woolf interval, Haldane-Anscombe 0.5 for zero cells."""
    a, b, c, d = k1 + 0.5, n1 - k1 + 0.5, k2 + 0.5, n2 - k2 + 0.5
    lor = np.log(a * d / (b * c))
    se = np.sqrt(1 / a + 1 / b + 1 / c + 1 / d)
    return np.exp(lor), np.exp(lor - z * se), np.exp(lor + z * se)


def add_reference_tests(frame, label=''):
    if frame is None or frame.empty:
        return frame
    out = frame.copy()
    for pop in ('EUR', 'EAS', 'SAS'):
        col = f'AF_1KG_{pop}'
        if col in out.columns:
            v = pd.to_numeric(out[col], errors='coerce')
            out[f'{col}_absent'] = v.isna()
            out[col] = v.fillna(0.0).round(4)

    k1 = (pd.to_numeric(out['KAZ_AF'], errors='coerce')
          * pd.to_numeric(out['n_called'], errors='coerce') * 2).round()
    n1 = pd.to_numeric(out['n_called'], errors='coerce') * 2
    ok = k1.notna() & n1.notna() & (n1 > 0)
    out['KAZ_AC'] = k1.fillna(0).astype(int)
    out['KAZ_AN'] = n1.fillna(0).astype(int)

    pv = {}
    for pop in TEST_POPS:
        col = f'AF_1KG_{pop}'
        if col not in out.columns:
            continue
        n2 = N_CHR_REF[pop]
        k2 = (pd.to_numeric(out[col], errors='coerce').fillna(0) * n2).round().astype(int)
        out[f'AC_1KG_{pop}'], out[f'AN_1KG_{pop}'] = k2, n2
        p, o_, l_, h_ = [], [], [], []
        for a, b, c, good in zip(out['KAZ_AC'], out['KAZ_AN'], k2, ok):
            if not good:
                p.append(np.nan); o_.append(np.nan); l_.append(np.nan); h_.append(np.nan)
                continue
            p.append(_st.fisher_exact([[a, b - a], [c, n2 - c]])[1])
            x, y, z_ = _or_ci(a, b, c, n2)
            o_.append(x); l_.append(y); h_.append(z_)
        out[f'p_{pop}'] = p
        out[f'OR_{pop}'] = np.round(o_, 3)
        out[f'OR_{pop}_lo'] = np.round(l_, 3)
        out[f'OR_{pop}_hi'] = np.round(h_, 3)
        pv[pop] = np.asarray(p, dtype=float)

    if pv:                                   # one BH family per table, all 2 x N tests
        keys = [k for k in TEST_POPS if k in pv]
        q = bh_q(np.concatenate([pv[k] for k in keys]))
        n = len(out)
        for i, pop in enumerate(keys):
            out[f'q_{pop}'] = np.round(q[i * n:(i + 1) * n], 6)

    glob = []
    have = all(f'AC_1KG_{p_}' in out.columns for p_ in TEST_POPS)
    for i, (a, b, good) in enumerate(zip(out['KAZ_AC'], out['KAZ_AN'], ok)):
        if not (good and have):
            glob.append(np.nan)
            continue
        idx = out.index[i]
        tab = [[a, b - a]] + [[int(out.at[idx, f'AC_1KG_{p_}']),
                               N_CHR_REF[p_] - int(out.at[idx, f'AC_1KG_{p_}'])]
                              for p_ in TEST_POPS]
        glob.append(_st.chi2_contingency(np.array(tab))[1])
    out['p_global'] = glob
    out['q_global'] = bh_q(np.asarray(glob, dtype=float))

    for pop in ('EUR', 'EAS'):
        f = f'AF_1KG_{pop}_absent'
        if f in out.columns and out[f].any():
            print(f'  {label}{pop}: {int(out[f].sum())} loci absent from the 1000G '
                  'panel, set to 0 and flagged')
    for pop in TEST_POPS:
        if f'q_{pop}' in out.columns:
            print(f'  {label}differ from {pop}: {int((out[f"q_{pop}"] < 0.05).sum())} '
                  f'of {int(out[f"q_{pop}"].notna().sum())} at BH q < 0.05')
    return out


df_trait = add_reference_tests(df_trait, 'trait loci, ')
df_pgx = add_reference_tests(df_pgx, 'pharmacogenomic loci, ')


def hwe_exact_p(n_hom_ref, n_het, n_hom_alt):
    """Exact HWE test, Wigginton, Cutler & Abecasis 2005 AJHG 76:887."""
    n_hom_ref, n_het, n_hom_alt = int(n_hom_ref), int(n_het), int(n_hom_alt)
    if min(n_hom_ref, n_het, n_hom_alt) < 0:
        return np.nan
    n = n_hom_ref + n_het + n_hom_alt
    if n == 0:
        return np.nan
    homc, homr = max(n_hom_ref, n_hom_alt), min(n_hom_ref, n_hom_alt)
    rare = 2 * homr + n_het
    if rare == 0:
        return 1.0
    probs = [0.0] * (rare + 1)
    mid = rare * (2 * n - rare) // (2 * n)
    if mid % 2 != rare % 2:
        mid += 1
    probs[mid] = 1.0
    total = 1.0
    h, hr, hc = mid, (rare - mid) // 2, n - mid - (rare - mid) // 2
    while h > 1:
        probs[h - 2] = probs[h] * h * (h - 1.0) / (4.0 * (hr + 1.0) * (hc + 1.0))
        total += probs[h - 2]
        hr, hc, h = hr + 1, hc + 1, h - 2
    h, hr, hc = mid, (rare - mid) // 2, n - mid - (rare - mid) // 2
    while h <= rare - 2 and hr > 0 and hc > 0:
        probs[h + 2] = probs[h] * 4.0 * hr * hc / ((h + 2.0) * (h + 1.0))
        total += probs[h + 2]
        hr, hc, h = hr - 1, hc - 1, h + 2
    target = probs[n_het] / total
    return min(1.0, sum(x / total for x in probs if x / total <= target + 1e-12))


def add_hwe(frame, label):
    """Exact HWE on autosomal diploid genotypes; chrX and hemizygous rows skipped."""
    if frame is None or frame.empty:
        return frame, pd.DataFrame()
    out = frame.copy()
    chrom = out['Chr'].astype(str).str.replace('^chr', '', regex=True)
    diploid = (chrom.isin([str(i) for i in range(1, 23)])
               & (out.get('n_hemi_alt', pd.Series(0, index=out.index)).fillna(0) == 0))
    p = pd.Series(np.nan, index=out.index, dtype=float)
    for i in out.index[diploid]:
        p.loc[i] = hwe_exact_p(out.at[i, 'n_hom_ref'], out.at[i, 'n_het'],
                               out.at[i, 'n_hom_alt'])
    out['HWE_exact_p'] = p
    out['HWE_q_BH'] = bh_q(p.to_numpy())
    out['HWE_flag'] = np.where(out['HWE_q_BH'] < 0.05, 'deviates', '')
    rep = out.loc[diploid, ['avsnp151', 'Curated_gene', 'Variant', 'n_hom_ref', 'n_het',
                            'n_hom_alt', 'n_called', 'HWE_exact_p', 'HWE_q_BH',
                            'HWE_flag']].copy()
    rep.insert(0, 'Set', label)
    n_dev = int((rep['HWE_flag'] == 'deviates').sum())
    print(f'  {label}: {len(rep)} autosomal loci tested for HWE, {n_dev} deviate at '
          'BH q < 0.05')
    return out, rep


df_trait, hwe_trait = add_hwe(df_trait, 'trait loci')
df_pgx, hwe_pgx = add_hwe(df_pgx, 'pharmacogenomic loci')
df_hwe = pd.concat([hwe_trait, hwe_pgx], ignore_index=True)

for _f, _n in ((df_trait, 'df_trait'), (df_pgx, 'df_pgx')):
    _miss = [c for c in ('KAZ_AF', 'AF_1KG_EUR', 'AF_1KG_EAS', 'q_EUR', 'q_EAS',
                         'HWE_exact_p') if c not in _f.columns]
    assert not _miss, f'{_n} is missing {_miss}'

df_identity = pd.concat([id_trait.assign(Panel='trait'), id_pgx.assign(Panel='pgx')],
                        ignore_index=True)
n_id_fail = int((df_identity['Status'] == 'FAIL').sum())

cols = ['avsnp151', 'Curated_gene', 'Variant', 'Gene_strand', 'Genomic_change',
        'Coding_change', 'Effect_allele', 'Effect_allele_is_REF', 'KAZ_AF',
        'KAZ_AF_CI_lower', 'KAZ_AF_CI_upper', 'AF_1KG_EUR', 'AF_1KG_EAS',
        'Site_FILTER', 'Identity_status']
for name, frame in (('trait loci', df_trait), ('pharmacogenomic loci', df_pgx)):
    if frame is None or frame.empty:
        continue
    print(f'\n{name}, polarised frequencies')
    display(frame[[c for c in cols if c in frame.columns]]
            .sort_values('Curated_gene').reset_index(drop=True))

## 9. Figure 5 - pathogenic and likely pathogenic variation

**a** carriers per gene, marker area is the number of reported variants in that gene;
**b** number of P/LP variants carried per individual; **c** ClinVar review status of the tier 1
set, with the reported threshold marked.

In [ ]:
# Fig. 5
PT, PT_GENE = FS, FS
INK, MUTED, LEADER, EXPECT = '#1A1A1A', '#C4CCD2', '#9AA5AD', '#B5502F'
MIN_CARRIERS = 3  
ROW_MM = 5.8

# a
gene = (df_rep.assign(_g=df_rep['Gene.refGene'].astype(str).str.split(';|,').str[0])
        .groupby('_g')
        .agg(carriers=('n_carriers', 'sum'), variants=('_g', 'size')))
sel = gene[gene['carriers'] >= MIN_CARRIERS].sort_values(['carriers', 'variants'])
cover = sel['carriers'].sum() / gene['carriers'].sum() * 100

fig_h = max(112, ROW_MM * len(sel) + 30)
fig = plt.figure(figsize=(FIG_W2 * MM, fig_h * MM))
gs = fig.add_gridspec(2, 2, width_ratios=[0.92, 1.08], wspace=0.38, hspace=0.62,
                      left=0.135, right=0.985, top=0.93, bottom=0.085)

ax = fig.add_subplot(gs[:, 0])
y = np.arange(len(sel))
ax.hlines(y, 0, sel['carriers'], color=LEADER, lw=0.8, linestyle=(0, (1, 2.2)), zorder=1)
ax.scatter(sel['carriers'], y, s=26, color=INK, edgecolor='white', lw=0.4, zorder=2)
for i, c in enumerate(sel['carriers']):
    ax.annotate(f'{int(c)}', (c, i), textcoords='offset points', xytext=(8, 0),
                va='center', ha='left', fontsize=PT)
ax.set_yticks(y)
ax.set_yticklabels([f'{g}  ({int(v)})' for g, v in zip(sel.index, sel['variants'])],
                   fontsize=PT_GENE, style='italic')
ax.set_xlim(0, sel['carriers'].max() * 1.22)
ax.set_ylim(-0.7, len(sel) - 0.3)
ax.set_xlabel('Carriers', fontsize=FS)
ax.tick_params(axis='x', labelsize=PT)
ax.tick_params(axis='y', length=0)
ax.spines['left'].set_visible(False)
ax.grid(True, axis='x')
ax.set_axisbelow(True)
ax.annotate('gene (variants)', (-0.02, 1.003), xycoords='axes fraction', ha='right',
            va='bottom', fontsize=FS, color=TEXT_DARK)
panel_label(ax, 'a', dy=9)

# b
ax = fig.add_subplot(gs[0, 1])
per_ind = ((dos_plp.loc[df_rep.index] > 0) & (plo_plp.loc[df_rep.index] > 0)).sum(axis=0)
vc = per_ind.value_counts().sort_index()
k, obs = vc.index.astype(int).to_numpy(), vc.to_numpy()
mean_v = float((k * obs).sum() / obs.sum())
p_var = ((dos_plp.loc[df_rep.index] > 0) &
         (plo_plp.loc[df_rep.index] > 0)).mean(axis=1).to_numpy(dtype=float)
pmf = np.array([1.0])
for p_i in p_var:
    pmf = np.convolve(pmf, [1.0 - p_i, p_i])
exp = pmf[:len(k)] * obs.sum()

ax.bar(k, obs, color=[MUTED if x == 0 else INK for x in k], width=0.62, **BAR_EDGE)
ax.plot(k, exp, marker='D', ms=3.4, lw=1.1, color=EXPECT, zorder=5,
        label='expected if independent')
for x, v in zip(k, obs):
    ax.text(x, v + obs.max() * 0.03, f'{v}', ha='center', va='bottom', fontsize=PT)
ax.set_xticks(k)
ax.set_ylim(0, obs.max() * 1.30)
ax.set_xlabel('Reported P/LP variants per individual\n(0 = non-carrier)',
              fontsize=FS)
ax.set_ylabel('Individuals', fontsize=FS)
ax.tick_params(labelsize=PT)
ax.grid(True, axis='y')
ax.set_axisbelow(True)
ax.legend(loc='upper right', fontsize=FS, frameon=False)
panel_label(ax, 'b')

e_pool, o_pool, acc_e, acc_o = [], [], 0.0, 0.0
for i in range(len(k) - 1, -1, -1):
    acc_e, acc_o = acc_e + exp[i], acc_o + obs[i]
    if acc_e >= 5:
        e_pool.insert(0, acc_e); o_pool.insert(0, acc_o); acc_e = acc_o = 0.0
if acc_e and e_pool:
    e_pool[0] += acc_e; o_pool[0] += acc_o
e_pool = np.asarray(e_pool) * (sum(o_pool) / sum(e_pool))
chi2, p_fit = _st.chisquare(o_pool, f_exp=e_pool, ddof=0)
ax.annotate(f'mean {mean_v:.2f}, goodness of fit p = {p_fit:.2f}', (0.98, 0.74),
            xycoords='axes fraction', ha='right', va='top', fontsize=FS,
            color=TEXT_DARK)

# c
ax = fig.add_subplot(gs[1, 1])
stars = pd.to_numeric(df_plp['ClinVar_stars'], errors='coerce').fillna(-1).astype(int)
sc = stars.value_counts().sort_index()
n_rep = int(sc[sc.index >= MIN_CLINVAR_STARS].sum())
ax.bar(range(len(sc)), sc.to_numpy(), width=0.62, **BAR_EDGE,
       color=[INK if i >= MIN_CLINVAR_STARS else MUTED for i in sc.index],
       hatch=[None if i >= MIN_CLINVAR_STARS else '////' for i in sc.index])
if MIN_CLINVAR_STARS in list(sc.index):
    ax.axvline(list(sc.index).index(MIN_CLINVAR_STARS) - 0.5, color=LEADER, lw=0.9,
               linestyle=(0, (2, 2)), zorder=3)
for x, v in enumerate(sc.to_numpy()):
    ax.text(x, v + sc.max() * 0.03, f'{v}', ha='center', va='bottom', fontsize=PT)
STAR_LABEL = {-1: 'n/a', 0: '0\nno\ncriteria', 1: '1\n1 sub-\nmitter',
              2: '2\nmultiple', 3: '3\nexpert', 4: '4\nguideline'}
ax.set_xticks(range(len(sc)))
ax.set_xticklabels([STAR_LABEL.get(i, str(i)) for i in sc.index], fontsize=FS,
                   linespacing=1.3)
ax.set_ylim(0, sc.max() * 1.18)
ax.set_xlabel('ClinVar review status', fontsize=FS)
ax.set_ylabel('Tier 1 variants', fontsize=FS)
ax.tick_params(labelsize=PT)
ax.grid(True, axis='y')
ax.set_axisbelow(True)
panel_label(ax, 'c')

SOURCE['19_SourceData_Fig5b'] = pd.DataFrame({
    'P/LP variants per individual': k, 'Individuals observed': obs,
    'Individuals expected if independent': np.round(exp, 3)})
SOURCE['20_SourceData_Fig5c'] = pd.DataFrame({
    'ClinVar stars': sc.index, 'Tier 1 variants': sc.to_numpy(),
    'Reported': sc.index >= MIN_CLINVAR_STARS})
save_figure(fig, f"Fig{MAIN_FIG['plp']}_PLP_variants")
print(f'  panel a (for the legend): {len(sel)} genes carry {cover:.0f}% of all carrier events')
print(f'  panel a: {len(sel)} genes with {MIN_CARRIERS}+ carriers, {cover:.0f}% of '
      f'carrier events, of {len(gene)} genes total')
print(f'  panel b: mean {mean_v:.2f}, {len(o_pool)} pooled classes, p = {p_fit:.3f}')
print(f'  panel c: {n_rep} of {len(df_plp)} at >= {MIN_CLINVAR_STARS} stars')
plt.show()

## 10. Fig. 4 - pharmacogenomic allele frequencies

Difference in effect-allele frequency between the cohort and 1000 Genomes EUR (circles) and
EAS (squares), with 95% Newcombe intervals; filled markers differ at BH q < 0.05. Panel **a**
is CPIC level A with a final assignment, panel **b** everything weaker; the panel names are
printed for the legend rather than drawn as titles. Loci whose site-level `FILTER` is not
`PASS` are omitted. The trait loci are a main-text table, not a figure.

In [ ]:
# Fig. 4

INK, RULE, FAINT, BAND = '#1A1A1A', '#3C3C3C', '#B0B0B0', '#4A4A4A'
REF_BLUE, REF_BRICK, NEUTRAL = '#2C6E9B', '#B5502F', '#A0A0A0'
COL.update({'cohort': INK, 'eas': REF_BLUE, 'eur': REF_BRICK, 'dark': RULE,
            'caption': BAND, 'neutral': NEUTRAL})
ROW_BAND = '#F3F5F7'
COHORT_LABEL = 'Kazakh'     
PT = FS
N_CHR_REF = {'EUR': 1006, 'EAS': 1008}
Q_CUT = 0.05

CAT_SHORT = {
    'Opioids and antidepressants': 'Opioids', 'Folate and methotrexate': 'Folate/MTX',
    'Platinum and taxanes': 'Platinum/tax.', 'Immunosuppression': 'Immunosuppr.',
    'Fluoropyrimidines': 'Fluoropyrim.', 'Antituberculosis': 'Anti-TB',
    'Anticoagulation': 'Anticoag.',
}
VAR_SHORT = {'HapB3 c.1129-5923C>G': 'HapB3 intronic'}

CAT_FIX = {'rs6025': 'Thrombophilia', 'rs1799963': 'Thrombophilia'}


def pgx_label(r, maxlen=22):
    v = VAR_SHORT.get(str(r['Variant']), str(r['Variant']))
    v = re.sub(r'\s*\(\*\d+\s+signature\)', '', v).strip()
    lab = f"{r['Curated_gene']} {v}"
    if len(lab) > maxlen:
        stripped = re.sub(r'\s*p\..*$', '', v).strip()
        if stripped:
            lab = f"{r['Curated_gene']} {stripped}"
    return lab if len(lab) <= maxlen else lab[:maxlen].rsplit(' ', 1)[0]


def _wilson(k, n, z=1.96):
    if not n:
        return (np.nan, np.nan)
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return max(0.0, c - h), min(1.0, c + h)


def _newcombe(k1, n1, k2, n2):
    """Newcombe hybrid score interval for a difference of two proportions.

    A plain normal interval collapses to zero width when a reference frequency is
    zero, which is exactly where several of these loci sit.
    """
    p1, p2 = k1 / n1, k2 / n2
    l1, u1 = _wilson(k1, n1)
    l2, u2 = _wilson(k2, n2)
    d = p1 - p2
    return d, d - np.sqrt((p1 - l1) ** 2 + (u2 - p2) ** 2), \
        d + np.sqrt((u1 - p1) ** 2 + (p2 - l2) ** 2)


def difference_frame(frame):
    rows = []
    for _, r in frame.dropna(subset=['KAZ_AF']).iterrows():
        n1 = int(r['n_called']) * 2
        k1 = int(round(r['KAZ_AF'] * n1))
        cat = CAT_FIX.get(str(r['avsnp151']), r['Category'])
        lvl = str(r.get('CPIC_level', 'none')).strip()
        rec = {'label': pgx_label(r), 'cat': CAT_SHORT.get(cat, cat),
               'level': lvl if lvl in ('A', 'B', 'C') else 'none',
               'tier': 'A' if (lvl == 'A' and
                               str(r.get('CPIC_status', '')) == 'final') else 'other'}
        for pop in ('EUR', 'EAS'):
            n2 = N_CHR_REF[pop]
            k2 = int(round(float(r[f'AF_1KG_{pop}']) * n2))
            d, lo, hi = _newcombe(k1, n1, k2, n2)
            rec[f'd_{pop}'], rec[f'lo_{pop}'], rec[f'hi_{pop}'] = d, lo, hi
            rec[f'sig_{pop}'] = float(r.get(f'q_{pop}', 1)) < Q_CUT
        rows.append(rec)
    d = pd.DataFrame(rows)
    d['_max'] = d[['d_EUR', 'd_EAS']].abs().max(axis=1)
    order = d.groupby('cat')['_max'].max().sort_values(ascending=False).index.tolist()
    d['_cat_rank'] = d['cat'].map({c: i for i, c in enumerate(order)})
    return d.sort_values(['_cat_rank', '_max'], ascending=[True, False]).reset_index(drop=True)


def draw_differences(ax, d, group_x=1.04):
    y = np.arange(len(d))
    starts = [0] + list(d['cat'].ne(d['cat'].shift()).to_numpy().nonzero()[0][1:]) + [len(d)]
    for i in range(len(starts) - 1):
        if i % 2 == 0:
            ax.axhspan(starts[i] - 0.5, starts[i + 1] - 0.5, color=ROW_BAND, lw=0, zorder=0)
    ax.axvline(0, color=COL['dark'], lw=0.8, zorder=2)
    for off, pop, colr, mk in ((-0.20, 'EUR', COL['eur'], 'o'), (0.20, 'EAS', COL['eas'], 's')):
        v = d[f'd_{pop}'].to_numpy(dtype=float)
        lo = d[f'lo_{pop}'].to_numpy(dtype=float)
        hi = d[f'hi_{pop}'].to_numpy(dtype=float)
        sig = d[f'sig_{pop}'].to_numpy(dtype=bool)
        ax.hlines(y + off, lo, hi, color=colr, lw=1.1, alpha=0.9, zorder=3)
        ax.scatter(v[sig], (y + off)[sig], s=21, marker=mk, color=colr, edgecolor='white',
                   lw=0.4, zorder=4)
        ax.scatter(v[~sig], (y + off)[~sig], s=21, marker=mk, facecolor='white',
                   edgecolor=colr, lw=0.9, zorder=4)
    for grp, sub in d.groupby('cat', sort=False):
        ax.text(group_x, float(np.mean(sub.index.to_numpy())), grp,
                transform=ax.get_yaxis_transform(), fontsize=PT, va='center',
                ha='left', color=INK)
    ax.set_yticks(y)
    ax.set_yticklabels(d['label'], fontsize=PT)
    ax.set_ylim(-0.5, len(d) - 0.5)
    ax.set_xlim(-0.42, 0.42)         
    ax.set_xticks([-0.4, -0.2, 0, 0.2, 0.4])   
    ax.set_xlabel('Difference in effect allele frequency\n'
                  f'({COHORT_LABEL} - reference)', fontsize=FS)
    ax.tick_params(axis='x', labelsize=PT)
    ax.tick_params(axis='y', length=0)
    ax.grid(True, axis='x', zorder=1)
    ax.set_axisbelow(True)
    for side in ('top', 'right', 'left'):
        ax.spines[side].set_visible(False)


def split_by_level(d):
    """Panel a, CPIC level A; panel b, everything weaker. The split is the point."""
    return [d[d['tier'] == t].reset_index(drop=True) for t in ('A', 'other')]


LAB_MM, PLOT_MM, GRP_MM, PAD_MM = 35, 32, 21, 2
BOT_MM, LEG_MM, TOP_MM = 15, 7, 8
ROW_PITCH_MM = 5.8

dd = difference_frame(df_pgx)
halves = split_by_level(dd)
rows_max = max(len(h) for h in halves)
fig_h = max(90, ROW_PITCH_MM * rows_max + BOT_MM + LEG_MM + TOP_MM)

fig = plt.figure(figsize=(FIG_W2 * MM, fig_h * MM))
col_mm = LAB_MM + PLOT_MM + GRP_MM
axs = [fig.add_axes([(PAD_MM + LAB_MM + k * col_mm) / FIG_W2, (BOT_MM + LEG_MM) / fig_h,
                     PLOT_MM / FIG_W2, (fig_h - BOT_MM - LEG_MM - TOP_MM) / fig_h])
       for k in range(2)]
TITLES = ['CPIC level A, reviewed in a guideline',
          'Provisional, weaker or no CPIC level']
for k, (ax, half) in enumerate(zip(axs, halves)):
    draw_differences(ax, half)
    print(f"  panel {'ab'[k]} (for the legend): {TITLES[k]}, n = {len(half)}")
    panel_label(ax, 'ab'[k], dx=-(LAB_MM + PAD_MM) * 72 / 25.4, dy=8)
for ax in axs:                     # both ends on both panels, or the panel split
    ax.annotate('lower', (0.0, 1.006), xycoords='axes fraction',
                ha='left', va='bottom', fontsize=PT, color=INK)
    ax.annotate('higher', (1.0, 1.006), xycoords='axes fraction',
                ha='right', va='bottom', fontsize=PT, color=INK)

proxy = [plt.Line2D([], [], marker='o', lw=0, color=COL['eur'], markersize=3.7),
         plt.Line2D([], [], marker='s', lw=0, color=COL['eas'], markersize=3.7),
         plt.Line2D([], [], marker='|', lw=1.0, color=RULE, markersize=0),
         plt.Line2D([], [], marker='o', lw=0, markerfacecolor='white',
                    markeredgecolor=RULE, markeredgewidth=0.9, markersize=3.7)]
plabels = ['vs 1000G EUR', 'vs 1000G EAS', '95% CI (Newcombe)',
           f'open: not significant (q >= {Q_CUT})']
assert len(proxy) == len(plabels)
fig.legend(proxy, plabels, loc='lower center', ncol=4, frameon=False, fontsize=PT,
           handletextpad=0.35, columnspacing=1.8, bbox_to_anchor=(0.5, 0.8 / fig_h))

SOURCE['21_SourceData_Fig4'] = dd.drop(columns=['_max', '_cat_rank']).rename(columns={
    'label': 'Locus', 'cat': 'Therapeutic area', 'level': 'CPIC level', 'tier': 'Panel'})
save_figure(fig, f"Fig{MAIN_FIG['pgx']}_pharmacogenomics")
n_sig = int((dd['sig_EUR'] | dd['sig_EAS']).sum())
print(f'  {len(dd)} loci, {n_sig} differ from at least one reference')
print('  CPIC levels: ' + ', '.join(f'{k} {v}' for k, v in
                                    dd['level'].value_counts().items()))
plt.show()

## 11. Supplementary Fig. 7 - population context

**a**, **b** cohort against 1000 Genomes EUR and EAS with the identity line, all curated loci;
loci that differ (BH q < 0.05) are filled, the rest open; **c** lactase persistence alleles,
reference panels hatched; **d** APOE diplotypes. v24: Supplementary Fig. 6.

Source data: Supplementary Data 4, sheets `06_Trait_loci`, `11_Pharmacogenomic_loci`,
`07_Lactase_persistence` and `09_APOE_diplotypes`.

In [ ]:
# a, b  cohort against 1000 Genomes EUR and EAS; the shaded band is the range the
#       cohort estimate can fall in given the two sample sizes alone, so points
#       outside it differ by more than sampling explains;
# c     lactase persistence alleles;
# d     APOE diplotypes.

def place_labels(fig, ax, rows, xcol, ycol, tagcol, max_n=5, flip_at=0.55,
                 avoid=(), pad=1.5):
    """Annotate points, dropping any label whose drawn box hits another.

    Thinning by the distance between points is not enough: the text is far
    wider than the marker, so two points 0.15 apart still collide through
    their labels. This measures the rendered boxes instead.
    """
    fig.canvas.draw()
    rend = fig.canvas.get_renderer()
    taken = [ax.transData.transform(p) for p in avoid]
    boxes, kept = [], 0
    for _, r in rows.iterrows():
        if kept >= max_n:
            break
        x, y = float(r[xcol]), float(r[ycol])
        for dx, ha in ((6, 'left'), (-6, 'right')):
            for dy in (5, -10):
                if x < flip_at and ha == 'right':
                    continue
                if x >= flip_at and ha == 'left':
                    continue
                t = ax.annotate(str(r[tagcol]), (x, y), textcoords='offset points',
                                xytext=(dx, dy), ha=ha, fontsize=FS,
                                color=TEXT_DARK)
                bb = t.get_window_extent(rend).expanded(1 + pad / 20, 1 + pad / 8)
                clash = any(bb.overlaps(b) for b in boxes) or \
                    any(bb.contains(px, py) for px, py in taken)
                if clash or not ax.bbox.contains(bb.x0, bb.y0) \
                        or not ax.bbox.contains(bb.x1, bb.y1):
                    t.remove()
                    continue
                boxes.append(bb)
                kept += 1
                break
            else:
                continue
            break
    return kept


N_CHR_REF = {'AF_1KG_EUR': 1006, 'AF_1KG_EAS': 1008}   # 1000G Phase 3, 503 and 504 individuals
ROW_BAND = '#F3F5F7'
OUTLIER = REF_BRICK
MAX_LABELS, LABEL_FLIP = 5, 0.55
Q_CUT = 0.05

PAD_MM, LAB_MM, PANEL_MM, GAP_MM = 3, 15, 64, 21
TOPROW_MM, MIDGAP_MM, BOTROW_MM = 55, 20, 44
BOT_MM, LEG_MM, TOP_MM = 13, 8, 8

fig_h = BOT_MM + BOTROW_MM + MIDGAP_MM + TOPROW_MM + LEG_MM + TOP_MM
fig = plt.figure(figsize=(FIG_W2 * MM, fig_h * MM))
x0 = [(PAD_MM + LAB_MM) / FIG_W2,
      (PAD_MM + LAB_MM + PANEL_MM + GAP_MM) / FIG_W2]
y_top = (BOT_MM + LEG_MM + BOTROW_MM + MIDGAP_MM) / fig_h
y_bot = (BOT_MM + LEG_MM) / fig_h
axs = np.array([[fig.add_axes([x0[k], y_top, PANEL_MM / FIG_W2, TOPROW_MM / fig_h])
                 for k in range(2)],
                [fig.add_axes([x0[k], y_bot, PANEL_MM / FIG_W2, BOTROW_MM / fig_h])
                 for k in range(2)]], dtype=object)

both = pd.concat([df_trait.assign(_set='trait'), df_pgx.assign(_set='pgx')],
                 ignore_index=True)
if both['Curated_gene'].duplicated().any():
    dup = both['Curated_gene'].duplicated(keep=False)
    both['_tag'] = np.where(dup,
                            both['Curated_gene'] + ' ' + both['avsnp151'].astype(str),
                            both['Curated_gene'])
else:
    both['_tag'] = both['Curated_gene']

# a, b
for k, (ref_col, ref_name, qcol) in enumerate(
        [('AF_1KG_EUR', '1000G EUR', 'q_EUR'), ('AF_1KG_EAS', '1000G EAS', 'q_EAS')]):
    ax = axs[0, k]
    dd = both.dropna(subset=['KAZ_AF', ref_col]).copy()
    if dd.empty:
        ax.set_axis_off()
        continue
    n_ref = N_CHR_REF[ref_col]

    if qcol in dd.columns:
        dd['_out'] = pd.to_numeric(dd[qcol], errors='coerce').fillna(1) < Q_CUT
    else:
        _r = dd[ref_col].to_numpy(dtype=float)
        _k = dd['KAZ_AF'].to_numpy(dtype=float)
        dd['_out'] = np.abs(_k - _r) > 1.96 * np.sqrt(_r * (1 - _r)
                                                      * (1 / N_CHR_KZ + 1 / n_ref))
        print(f'  WARNING: {qcol} not found, colouring by the band instead; '
              'run the reference-test cell first')

    grid = np.linspace(0, 1, 201)
    half = 1.96 * np.sqrt(grid * (1 - grid) * (1 / N_CHR_KZ + 1 / n_ref))
    ax.fill_between(grid, np.clip(grid - half, 0, 1), np.clip(grid + half, 0, 1),
                    color=ROW_BAND, lw=0, zorder=0)
    ax.plot([0, 1], [0, 1], color=FAINT, lw=0.6, zorder=1)

    for s, mk in (('trait', 'o'), ('pgx', 'D')):
        sub = dd[dd['_set'] == s]
        o_ = sub['_out'].to_numpy(dtype=bool)
        ax.scatter(sub.loc[o_, ref_col], sub.loc[o_, 'KAZ_AF'], s=15, marker=mk, lw=0.35,
                   edgecolor='white', color=OUTLIER, zorder=3)
        ax.scatter(sub.loc[~o_, ref_col], sub.loc[~o_, 'KAZ_AF'], s=13, marker=mk, lw=0.7,
                   facecolor='white', edgecolor=INK, zorder=3)
    cand = (dd[dd['_out']]
            .assign(_d=lambda t: (t['KAZ_AF'] - t[ref_col]).abs())
            .sort_values('_d', ascending=False))
    place_labels(fig, ax, cand, ref_col, 'KAZ_AF', '_tag',
                 max_n=MAX_LABELS, flip_at=LABEL_FLIP,
                 avoid=list(zip(dd[ref_col], dd['KAZ_AF'])))
    print(f"  panel {'ab'[k]} (for the legend): {int(dd['_out'].sum())} of {len(dd)} loci "
          f"differ from {ref_name}")
    ax.set_xlim(-0.04, 1.04)
    ax.set_ylim(-0.04, 1.04)
    ax.set_xticks([0, 0.25, 0.5, 0.75, 1.0])
    ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
    ax.set_xlabel(f'{ref_name} frequency')
    ax.set_ylabel(f'{COHORT} frequency' if k == 0 else '')
    ax.grid(True, zorder=1)
    ax.set_axisbelow(True)
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
    panel_label(ax, 'ab'[k], dx=-(LAB_MM + PAD_MM) * 72 / 25.4, dy=3)

# c, lactase persistence
ax = axs[1, 0]
d = df_lactase[df_lactase['Observed'] == 'yes'].reset_index(drop=True)
SERIES = [('KAZ_AF', COL['cohort'], COHORT, None),
          ('1KG EUR', COL['eur'], '1000G EUR', '////'),
          ('1KG EAS', COL['eas'], '1000G EAS', '....')]
handles_c, labels_c = [], []
if not d.empty:
    x = np.arange(len(d))
    w = 0.8 / len(SERIES)
    for j_, (col, colr, nm, hat) in enumerate(SERIES):
        v = pd.to_numeric(d.get(col), errors='coerce').fillna(0.0).to_numpy(dtype=float)
        pos = x - 0.4 + w / 2 + j_ * w
        err = None
        if col == 'KAZ_AF':
            err = np.vstack([(d[col] - d['95% CI lower']).clip(lower=0),
                             (d['95% CI upper'] - d[col]).clip(lower=0)])
        nz = v > 0
        if nz.any():
            ax.bar(pos[nz], v[nz], w, yerr=(err[:, nz] if err is not None else None),
                   color=colr, error_kw={'lw': 0.6, 'capsize': 1.5,
                                         'ecolor': COL['dark']},
                   hatch=hat, edgecolor='white', linewidth=0.4, zorder=3)
        for xi in pos[~nz]:                       # a measured zero, not a missing bar
            ax.plot([xi - w / 2, xi + w / 2], [0, 0], color=colr, lw=2.2,
                    solid_capstyle='butt', zorder=4)
            ax.annotate('0', (xi, 0), textcoords='offset points', xytext=(0, 3),
                        ha='center', va='bottom', fontsize=FS, color=colr)
        handles_c.append(mpl.patches.Patch(facecolor=colr, edgecolor='white', hatch=hat))
        labels_c.append(nm)
    ax.set_xticks(x)
    ax.set_xticklabels(d['Variant'], rotation=25, ha='right', fontsize=FS)
    ax.set_ylabel('Persistence allele\nfrequency')
    ax.set_ylim(0, 0.62)
    ax.grid(True, axis='y', zorder=1)
    ax.set_axisbelow(True)
for side in ('top', 'right'):
    ax.spines[side].set_visible(False)
panel_label(ax, 'c', dx=-(LAB_MM + PAD_MM) * 72 / 25.4, dy=3)

# d, APOE diplotypes
ax = axs[1, 1]
if not df_apoe_geno.empty:
    a = df_apoe_geno.sort_values('Individuals').reset_index(drop=True)
    yy = np.arange(len(a))
    for i in yy:
        if i % 2 == 0:
            ax.axhspan(i - 0.5, i + 0.5, color=ROW_BAND, lw=0, zorder=0)
    ax.barh(yy, a['Individuals'], color=COL['cohort'], height=0.6, zorder=3,
            edgecolor='white', linewidth=0.4)
    for yi, vi in zip(yy, a['Individuals']):
        ax.annotate(f'{int(vi)}  ({vi / N_COHORT * 100:.1f}%)', (vi, yi),
                    textcoords='offset points', xytext=(4, 0), va='center',
                    fontsize=FS, color=TEXT_DARK)
    ax.set_yticks(yy)
    ax.set_yticklabels(a['APOE diplotype'], fontsize=FS)
    ax.set_ylim(-0.5, len(a) - 0.5)
    ax.set_xlim(0, a['Individuals'].max() * 1.42)
    ax.set_xlabel('Individuals')
    ax.grid(True, axis='x', zorder=1)
    ax.set_axisbelow(True)
    ax.tick_params(axis='y', length=0)
for side in ('top', 'right', 'left'):
    ax.spines[side].set_visible(False)
panel_label(ax, 'd', dx=-(LAB_MM + PAD_MM) * 72 / 25.4, dy=3)

proxy = [plt.Line2D([], [], marker='o', lw=0, markerfacecolor='white',
                    markeredgecolor=INK, markeredgewidth=0.7, markersize=3.6),
         plt.Line2D([], [], marker='D', lw=0, markerfacecolor='white',
                    markeredgecolor=INK, markeredgewidth=0.7, markersize=3.4),
         plt.Line2D([], [], marker='s', lw=0, color=OUTLIER, markersize=3.6)]
plabels = ['trait locus', 'pharmacogenomic locus', f'filled: differs, BH q < {Q_CUT}']
assert len(proxy) == len(plabels), 'legend handles and labels must match'
axs[0, 0].legend(proxy, plabels, loc='upper left', frameon=False, fontsize=FS,
                 handletextpad=0.35, labelspacing=0.35, borderpad=0.2)
fig.legend(handles_c, labels_c, loc='lower center', ncol=len(labels_c), frameon=False,
           fontsize=FS, handletextpad=0.35, columnspacing=1.3, handlelength=1.4,
           bbox_to_anchor=((PAD_MM + LAB_MM + PANEL_MM / 2) / FIG_W2, 0.6 / fig_h))

for ref_col, nm in (('AF_1KG_EUR', 'EUR'), ('AF_1KG_EAS', 'EAS')):
    n_drop = int(both['KAZ_AF'].notna().sum()
                 - both.dropna(subset=['KAZ_AF', ref_col]).shape[0])
    print(f'  {nm}: {n_drop} loci without a reference frequency, not plotted')
save_figure(fig, f"Supplementary_Fig_{SUPP_NUM['fig_population_context']}_population_context")
plt.show()

## 12. Export

In [ ]:
XLSX = os.path.join(OUT_DIR, f"Supplementary_Data_{SUPP_NUM['data']}_biomedical_variation.xlsx")

DISPLAY_COLS = [
    'Chr', 'Start', 'Ref', 'Alt', 'Gene.refGene', 'ExonicFunc.refGene',
    'Transcript', 'Exon', 'HGVS_c', 'HGVS_p', 'HGVS_source', 'HGVS_requires_curation',
    'HGVS_annovar_unverified', 'avsnp151', 'CLNSIG', 'CLNDN', 'ClinVar_stars',
    'ClinVar_side_assertion', 'Inheritance', 'Reference_note',
    AF_COL, 'AF_gt', 'n_het', 'n_hom_alt', 'n_hemi_alt', 'n_hom_ref',
    'n_carriers', 'n_called', 'genotyping_rate', 'Low_call_rate',
    'VCF_multiallelic', 'Allele_index_unresolved', 'Count_discordant',
    'Site_FILTER', 'Gene_strand', 'Genomic_change', 'Coding_change', 'Identity_status',
    'Effect_allele', 'Effect_allele_is_REF', 'KAZ_AF',
    'KAZ_AF_CI_lower', 'KAZ_AF_CI_upper',
    'Effect_carriers (n)', 'Effect_homozygotes (n)', 'Panel',
]

def trim(frame, extra=()):
    if frame is None or getattr(frame, 'empty', True):
        return frame
    cols = [c for c in list(DISPLAY_COLS) + list(extra) if c in frame.columns]
    cols += [c for c in frame.columns
             if c.startswith(('AF_1KG', 'AF_gnomAD', 'gnomAD_source', 'KAZ_AC',
                             'KAZ_AN', 'AC_1KG', 'AN_1KG', 'p_', 'q_', 'OR_', 'HWE_'))
             and c not in cols]
    return frame[cols]

session = pd.DataFrame([
    ('Generated', datetime.now().strftime('%Y-%m-%d %H:%M:%S')),
    ('Cohort', f'{COHORT}, n = {N_COHORT} ({N_FEMALE} female, {N_MALE} male)'),
    ('Input file', os.path.basename(INPUT_FILE)),
    ('Reference', 'GRCh37/hg19'),
    ('Variant calling', f'{CALLER_SAMPLE} per sample, {CALLER_JOINT} joint genotyping'),
    ('Annotation', ANNOVAR_BUILD),
    ('ClinVar release', CLINVAR_RELEASE),
    ('ClinVar P/LP rule', 'token match on CLNSIG; Benign, Likely_benign, '
                          'Uncertain_significance and Conflicting excluded'),
    ('ClinVar review status reported', f'>= {MIN_CLINVAR_STARS} stars'),
    ('ACMG SF version', 'v3.3, 84 genes'),
    ('Excluded, repeat tracts', f'{len(REPEAT_TRACTS_HG19)} loci'),
    ('Excluded, paralogue-bearing genes', ', '.join(sorted(PARALOGUE_GENES))),
    ('Excluded, other', 'star alleles, diplotypes, HLA alleles'),
    ('Genotype ploidy', 'haploid calls contribute one chromosome to the allele number'),
    ('Reference panels', '1000 Genomes Phase 3; gnomAD v2.1.1, genomes for non-coding '
                         'and exomes for coding sites'),
    ('Reference frequencies', 'polarised to the effect allele; a value absent from the '
                              'per-population panel is a zero count and is flagged in '
                              'AF_1KG_*_absent'),
    ('Population comparison', 'two-sided Fisher exact against 1000G EUR (1,006 '
                              'chromosomes) and EAS (1,008), one Benjamini-Hochberg '
                              'family per table over all 2 x N tests'),
    ('Hardy-Weinberg', 'exact test (Wigginton 2005), autosomal diploid genotypes only'),
    ('Intervals', 'Wilson score, 95%'),
    ('Detection power', f'80% power to observe at least one carrier at an allele '
                        f'frequency of {af_80*100:.3f}% or higher'),
    ('Nomenclature', f'{len(CURATED_HGVS)} curated HGVS entries keyed on coordinate and '
                     'alleles; unresolved rows flagged HGVS_requires_curation'),
    ('Rows loaded', f'{len(df_raw):,}'),
    ('Tier 1 P/LP variants', f'{n_plp_all:,}'),
    ('Reported P/LP variants', f'{n_plp_reported:,}'),
    ('Python', sys.version.split()[0]),
    ('Platform', platform.platform()),
    ('numpy', np.__version__),
    ('pandas', pd.__version__),
    ('scipy', scipy.__version__),
    ('matplotlib', mpl.__version__),
], columns=['Key', 'Value'])

SHEETS = [
    ('01_Disease_panels',        lambda: panel_summary,
     'the curated disease panels, gene counts and composition'),
    ('02_Carrier_frequency',     lambda: df_carriers,
     'carrier frequency per panel with Wilson 95% intervals'),
    ('03_PLP_reported',          lambda: trim(df_panel_cmp),
     'P/LP variants in panel genes at the reporting threshold'),
    ('04_PLP_all_tier1',         lambda: trim(df_plp),
     'every tier 1 variant including one-star records, the sensitivity set'),
    ('05_ACMG_SF_v33',           lambda: trim(df_acmg, extra=['ACMG_gene', 'New_in_v3.3',
                                                              'ACMG_v33_footnote']),
     'variants in ACMG Secondary Findings v3.3 genes with the v3.3 footnotes'),
    ('06_Trait_loci',            lambda: trim(df_trait, extra=['Curated_gene', 'Variant',
                                                               'Category', 'Note']),
     'phenotype-associated loci, polarised frequencies, reference tests'),
    ('07_Lactase_persistence',   lambda: df_lactase,
     'the five persistence alleles including those not observed'),
    ('08_LCT_marker_concordance', lambda: df_lp_concord,
     'rs4988235 against rs182549 genotype by genotype'),
    ('09_APOE_diplotypes',       lambda: df_apoe_geno,
     'APOE diplotype counts assigned from both determinant sites'),
    ('10_APOE_alleles',          lambda: df_apoe,
     'epsilon2, epsilon3 and epsilon4 frequencies with intervals'),
    ('11_Pharmacogenomic_loci',  lambda: trim(df_pgx, extra=['Curated_gene', 'Variant',
                                                             'Category', 'Function',
                                                             'CPIC_level', 'CPIC_status',
                                                             'CPIC_drug', 'Note']),
     'pharmacogenomic loci with the CPIC pair, level and review status'),
    ('12_PGx_not_observed',      lambda: df_pgx_absent,
     'queried pharmacogenomic alleles absent from the callset, with upper bounds'),
    ('13_Hardy_Weinberg',        lambda: df_hwe,
     'exact Hardy-Weinberg test for every autosomal curated locus'),
    ('14_Variant_identity_QC',   lambda: df_identity,
     'coordinate, allele, effect allele and frequency check per locus'),
    ('15_HGVS_curation',         lambda: df_curated_check,
     'curated nomenclature entries and their verdicts'),
    ('16_Genotype_concordance',  lambda: (qc_plp if isinstance(qc_plp, pd.DataFrame)
                                          else pd.DataFrame()),
     'genotype parsing consistency check'),
    ('17_Excluded_variants',     lambda: (df_excluded[[c for c in
                                          ['Chr', 'Start', 'Ref', 'Alt', 'Gene.refGene',
                                           'ExonicFunc.refGene', 'avsnp151', 'CLNSIG',
                                           'Exclusion_reason']
                                          if c in df_excluded.columns]]
                                          if not df_excluded.empty else pd.DataFrame()),
     'variants excluded before analysis, with the reason for each'),
    ('18_Methods_and_versions',  lambda: session,
     'cohort, software, database releases, thresholds and decision rules'),
] + [(name, (lambda f=frame: f), desc) for name, frame, desc in (
    ('19_SourceData_Fig5b', SOURCE.get('19_SourceData_Fig5b'),
     f"source data, Fig. {MAIN_FIG['plp']}b: observed and expected individuals by number of P/LP variants"),
    ('20_SourceData_Fig5c', SOURCE.get('20_SourceData_Fig5c'),
     f"source data, Fig. {MAIN_FIG['plp']}c: tier 1 variants by ClinVar review status"),
    ('21_SourceData_Fig4', SOURCE.get('21_SourceData_Fig4'),
     f"source data, Fig. {MAIN_FIG['pgx']}: frequency differences with Newcombe intervals"),
)]

built, skipped = [], []
for name, make, desc in SHEETS:
    try:
        frame = make()
    except NameError as exc:
        skipped.append((name, f'not built: {exc}'))
        continue
    if frame is None or getattr(frame, 'empty', True):
        skipped.append((name, 'empty'))
        continue
    built.append((name, frame, desc))

readme = pd.DataFrame(
    [{'Sheet': n, 'Rows': f.shape[0], 'Columns': f.shape[1], 'Contents': d}
     for n, f, d in built])

from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter

HEAD_FILL = PatternFill(fill_type='solid', start_color='FFDDEBF7', end_color='FFDDEBF7')
HEAD_FONT = Font(name='Arial', size=10, bold=True, color='FF1F3864')
BODY_FONT = Font(name='Arial', size=10)
RULE_B = Border(bottom=Side(style='thin', color='FFBFBFBF'))

with pd.ExcelWriter(XLSX, engine='openpyxl') as xl:
    readme.to_excel(xl, sheet_name='00_README', index=False)
    for name, frame, _ in built:
        frame.to_excel(xl, sheet_name=name[:31], index=False)
    for ws in xl.book.worksheets:
        ws.freeze_panes = 'A2'
        for row in ws.iter_rows(min_row=2):
            for c in row:
                c.font = BODY_FONT
                c.alignment = Alignment(horizontal='center', vertical='center')
        for c in ws[1]:
            c.font, c.fill, c.border = HEAD_FONT, HEAD_FILL, RULE_B
            c.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        ws.row_dimensions[1].height = 30
        for col in ws.iter_cols(max_col=ws.max_column):
            width = max((len(str(c.value)) for c in col if c.value is not None), default=8)
            ws.column_dimensions[get_column_letter(col[0].column)].width = min(max(width + 3, 10), 44)

print(f'saved {XLSX}')
print(f'  00_README + {len(built)} data sheets')
for n, f, _ in built:
    print(f'    {n:26} {f.shape[0]:>5} rows x {f.shape[1]:>3} cols')
if skipped:
    print(f'  {len(skipped)} sheet(s) NOT written, the numbering will have gaps:')
    for n, why in skipped:
        print(f'    {n:26} {why}')

In [ ]:
from docx import Document
from docx.shared import Pt, Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT

def pick(df, cols):
    if df is None or getattr(df, 'empty', True):
        return None
    return df[[c for c in cols if c in df.columns]].copy()

SUPPL_TABLES = [
    ('tab_trait', 'Trait-associated and adaptive loci: cohort effect-allele frequencies with '
        'Wilson 95% confidence intervals and two-sided Fisher exact tests against the '
        '1000 Genomes European and East Asian panels, Benjamini-Hochberg adjusted.',
     pick(df_trait, ['Curated_gene', 'Variant', 'avsnp151', 'Effect_allele', 'KAZ_AF',
                     'KAZ_AF_CI_lower', 'KAZ_AF_CI_upper', 'Effect_carriers (n)',
                     'Effect_homozygotes (n)', 'AF_1KG_EUR', 'q_EUR',
                     'AF_1KG_EAS', 'q_EAS', 'Category'])),

    ('tab_pgx', 'Complete pharmacogenomic panel: loci observed in the cohort with their CPIC '
        'gene-drug pair, level and review status.',
     pick(df_pgx, ['Curated_gene', 'Variant', 'avsnp151', 'Effect_allele', 'KAZ_AF',
                   'KAZ_AF_CI_lower', 'KAZ_AF_CI_upper', 'Effect_carriers (n)',
                   'Effect_homozygotes (n)', 'AF_1KG_EUR', 'q_EUR', 'AF_1KG_EAS',
                   'q_EAS', 'CPIC_level', 'CPIC_status', 'CPIC_drug'])),

    ('tab_pgx_absent', 'Queried pharmacogenomic alleles not observed in the cohort, with the '
        'upper bound on the allele frequency for the number of chromosomes assayed.',
     pick(df_pgx_absent, ['Curated_gene', 'Variant', 'avsnp151', 'Effect_allele',
                          'n_called', 'Upper_bound', 'CPIC_level', 'CPIC_drug', 'Note'])),

    ('tab_excluded', 'Variants excluded before analysis, with the reason for each exclusion.',
     pick(df_excluded, ['Chr', 'Start', 'Ref', 'Alt', 'Gene.refGene',
                        'ExonicFunc.refGene', 'avsnp151', 'CLNSIG', 'Exclusion_reason'])),

    ('tab_hwe', 'Hardy-Weinberg equilibrium testing of every autosomal curated locus, exact '
        'test with Benjamini-Hochberg correction.',
     pick(df_hwe, ['Set', 'Curated_gene', 'Variant', 'avsnp151', 'n_hom_ref', 'n_het',
                   'n_hom_alt', 'n_called', 'HWE_p', 'HWE_q'])),

    ('tab_acmg', 'Variants in ACMG Secondary Findings v3.3 genes, with the reporting footnote '
         'applicable to each gene-disease pair.',
     pick(df_acmg, ['ACMG_gene', 'Variant', 'HGVS_c', 'HGVS_p', 'avsnp151', 'CLNSIG',
                    'ClinVar_stars', 'n_carriers', 'Inheritance', 'ACMG_v33_footnote',
                    'New_in_v3.3'])),
]
SUPPL_TABLES = sorted(SUPPL_TABLES, key=lambda t: SUPP_NUM[t[0]])

def _fmt(v):
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return ''
    if isinstance(v, (bool, np.bool_)):
        return 'yes' if v else 'no'
    if isinstance(v, (int, np.integer)):
        return f'{int(v):,}'
    if isinstance(v, (float, np.floating)):
        return f'{v:.3g}' if abs(v) < 0.01 else f'{v:,.4f}'.rstrip('0').rstrip('.')
    return str(v)

def add_table(doc, df, caption, tag, landscape_hint=9):
    p = doc.add_paragraph()
    r = p.add_run(f'{tag}  {caption}')
    r.bold = True
    r.font.size = Pt(9)
    r.font.name = 'Arial'
    p.paragraph_format.space_after = Pt(4)
    size = Pt(7) if len(df.columns) > landscape_hint else Pt(8)
    t = doc.add_table(rows=1, cols=len(df.columns))
    t.style = 'Table Grid'
    t.alignment = WD_TABLE_ALIGNMENT.CENTER
    for j, col in enumerate(df.columns):
        c = t.rows[0].cells[j]
        c.text = ''
        run = c.paragraphs[0].add_run(str(col))
        run.bold = True; run.font.size = size; run.font.name = 'Arial'
        c.paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.CENTER
    for _, row in df.iterrows():
        cells = t.add_row().cells
        for j, v in enumerate(row):
            cells[j].text = ''
            run = cells[j].paragraphs[0].add_run(_fmt(v))
            run.font.size = size; run.font.name = 'Arial'
            cells[j].paragraphs[0].alignment = (
                WD_ALIGN_PARAGRAPH.LEFT if j == 0 else WD_ALIGN_PARAGRAPH.CENTER)
    doc.add_paragraph().paragraph_format.space_after = Pt(10)

doc = Document()
for s in doc.sections:
    s.left_margin = s.right_margin = Inches(0.6)
    s.top_margin = s.bottom_margin = Inches(0.6)
h = doc.add_paragraph()
hr = h.add_run('Supplementary Tables generated 04')
hr.bold = True
hr.font.size = Pt(12)
hr.font.name = 'Arial'

written = 0
for key, cap, df in SUPPL_TABLES:
    num = SUPP_NUM[key]
    if df is None or df.empty:
        print(f'  Supplementary Table {num}: not available, skipped')
        continue
    add_table(doc, df, cap, f'Supplementary Table {num}.')
    written += 1
    print(f'  Supplementary Table {num}: {df.shape[0]} rows x {df.shape[1]} cols')

DOCX = os.path.join(OUT_DIR, 'Supplementary_Tables_04.docx')
doc.save(DOCX)
print(f'\nsaved {os.path.basename(DOCX)}: {written} tables')